# Román et al. and reliability-qualified recovery profiles for Samar–Leyte

This notebook compares two regional nighttime-light processing approaches for Typhoon Haiyan:

1. **Román-style profile:** whole-region Black Marble DNB-BRDF, four-day multi-date aggregation, and recovery relative to pre-hurricane radiance.
2. **Reliability-qualified profile:** direct DNB-BRDF observations with `MQF == 0`, fixed GHSL G7 settlement support, daily spatial-completeness filtering, 95th-percentile clipping, a centred 30-day moving average, and pre-event standardisation.

Both profiles are compared with Samar–Leyte NGCP 1 AM electricity load.

Román et al. specify four-day **multi-date aggregation**, not a four-day moving average. Accordingly, the first profile uses non-overlapping four-day mean composites. The paper defines \(NTL_0\) as pre-hurricane radiance but does not report a transferable baseline date range; this implementation uses the four complete days immediately before Haiyan.

Only time-series profiles are evaluated here. NDWE and CHI are excluded.

Reference: [Román et al. (2019)](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0218883).

In [ ]:
# ============================================================
# 1. IMPORTS AND SETTINGS
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr

from rasterio.enums import Resampling

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display



In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_DIR / "datasets"
VNP46_DIR = DATA_DIR / "VNP46"
PROCESSED_DIR = VNP46_DIR / "processed"

A2_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"

GHSL_CANDIDATES = [
    VNP46_DIR / "GHSL_SMOD_E2015.tif",
    DATA_DIR / "ghsl" / "GHSL_SMOD_E2015.tif",
]

GHSL_PATH = next(
    (path for path in GHSL_CANDIDATES if path.exists()),
    GHSL_CANDIDATES[0],
)

NGCP_CANDIDATES = [
    DATA_DIR / "NGCP_hourly_load.xlsx",
    PROJECT_DIR / "Datasets" / "NGCP_hourly_load.xlsx",
    PROJECT_DIR.parent / "Datasets" / "NGCP_hourly_load.xlsx",
    DATA_DIR / "NGCP" / "NGCP_hourly_load.xlsx",
]

NGCP_PATH = next(
    (path for path in NGCP_CANDIDATES if path.exists()),
    NGCP_CANDIDATES[0],
)

NGCP_SHEET = "LEY-SAM HOURLY LOAD 2013-2024"

# ------------------------------------------------------------
# Event windows
# ------------------------------------------------------------

EVENT_DATE = pd.Timestamp("2013-11-08")

ANALYSIS_START = EVENT_DATE - pd.Timedelta(days=180)
ANALYSIS_END = EVENT_DATE + pd.Timedelta(days=365)

PROFILE_END = EVENT_DATE + pd.Timedelta(days=179)

ROMAN_BASELINE_START = EVENT_DATE - pd.Timedelta(days=4)
ROMAN_BASELINE_END = EVENT_DATE - pd.Timedelta(days=1)

# ------------------------------------------------------------
# VNP46A2 bands
# ------------------------------------------------------------

DNB_BAND = "DNB_BRDF_Corrected_NTL"
MQF_BAND = "Mandatory_Quality_Flag"

# ------------------------------------------------------------
# Román-style settings
# ------------------------------------------------------------

ROMAN_BLOCK_DAYS = 4

# ------------------------------------------------------------
# Reliability-qualified settings
# ------------------------------------------------------------

GHSL_G7_CLASSES = (23, 30)
RQ_VALID_PCT = 10.0
RQ_CLIP_PERCENTILE = 95.0
RQ_MOVING_WINDOW = 30
RQ_MOVING_MIN_OBS = 10
RQ_MIN_BASELINE_OBS = 1

# ------------------------------------------------------------
# Basic data validity
# ------------------------------------------------------------

KNOWN_FILL_VALUES = (
    -9999.0,
    65535.0,
    6553.5,
)

# ------------------------------------------------------------
# Plot style
# ------------------------------------------------------------

ROMAN_NTL_COLOR = "#F20D0D"
RQ_NTL_COLOR = "#059669"
NGCP_COLOR = "#111111"

EVENT_LINE_COLOR = "#2563EB"
STAGE_LINE_COLOR = "#94A3B8"

print("VNP46A2:", A2_ZARR_PATH)
print("GHSL:", GHSL_PATH)
print("NGCP:", NGCP_PATH)

In [ ]:
# ============================================================
# 2. LOAD AND CLEAN VNP46A2
# ============================================================

for label, path in {
    "VNP46A2": A2_ZARR_PATH,
    "GHSL": GHSL_PATH,
    "NGCP": NGCP_PATH,
}.items():
    if not path.exists():
        raise FileNotFoundError(
            f"{label} was not found:\n{path}"
        )


def open_zarr_safely(path):
    try:
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks="auto",
            mask_and_scale=True,
            decode_cf=True,
        )
    except (ImportError, ModuleNotFoundError, ValueError):
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks=None,
            mask_and_scale=True,
            decode_cf=True,
        )


def standardise_date_dimension(ds):
    if "date" not in ds.variables:
        raise KeyError(
            f"No `date` variable found. Variables: {list(ds.variables)}"
        )

    if "date" not in ds.coords:
        ds = ds.set_coords("date")

    observation_dim = ds["date"].dims[0]

    dates = pd.DatetimeIndex(
        pd.to_datetime(ds["date"].values)
    ).normalize()

    ds = ds.assign_coords(
        date=(observation_dim, dates.values)
    )

    if observation_dim != "date":
        ds = ds.swap_dims({observation_dim: "date"})

    return ds.sortby("date")


def prepare_spatial_metadata(ds):
    ds = ds.rio.set_spatial_dims(
        x_dim="x",
        y_dim="y",
        inplace=False,
    )

    if ds.rio.crs is None and "spatial_ref" in ds.variables:
        spatial_attrs = ds["spatial_ref"].attrs

        stored_crs = (
            spatial_attrs.get("crs_wkt")
            or spatial_attrs.get("spatial_ref")
        )

        if stored_crs is not None:
            ds = ds.rio.write_crs(
                stored_crs,
                inplace=False,
            )

    if ds.rio.crs is None:
        x_min = float(ds["x"].min())
        x_max = float(ds["x"].max())
        y_min = float(ds["y"].min())
        y_max = float(ds["y"].max())

        coordinates_are_lonlat = (
            -180 <= x_min <= 180
            and -180 <= x_max <= 180
            and -90 <= y_min <= 90
            and -90 <= y_max <= 90
        )

        if coordinates_are_lonlat:
            ds = ds.rio.write_crs(
                "EPSG:4326",
                inplace=False,
            )
        else:
            raise ValueError(
                "The VNP46A2 CRS could not be recovered."
            )

    return ds


def clean_radiance(values):
    """
    Remove fill values and invalid radiance.

    This is basic data cleaning applied to both methods, not an
    additional reliability filter.
    """
    cleaned = values.astype("float32")

    cleaned = cleaned.where(
        np.isfinite(cleaned)
    )

    fill_values = list(KNOWN_FILL_VALUES)

    for source in (
        values.attrs,
        values.encoding,
    ):
        for key in (
            "_FillValue",
            "missing_value",
        ):
            fill_value = source.get(key)

            if fill_value is not None:
                fill_values.append(fill_value)

    for fill_value in fill_values:
        try:
            fill_value = float(fill_value)

            if np.isfinite(fill_value):
                cleaned = cleaned.where(
                    ~np.isclose(
                        cleaned,
                        fill_value,
                    )
                )
        except (TypeError, ValueError):
            continue

    # Negative DNB radiance is not retained in either profile.
    cleaned = cleaned.where(cleaned >= 0)

    return cleaned


a2 = prepare_spatial_metadata(
    standardise_date_dimension(
        open_zarr_safely(A2_ZARR_PATH)
    )
)

a2 = a2.sel(
    date=slice(
        ANALYSIS_START,
        ANALYSIS_END,
    )
)

missing_bands = [
    band
    for band in (
        DNB_BAND,
        MQF_BAND,
    )
    if band not in a2.data_vars
]

if missing_bands:
    raise KeyError(
        f"Missing A2 bands: {missing_bands}\n"
        f"Available bands: {list(a2.data_vars)}"
    )

dnb = clean_radiance(
    a2[DNB_BAND]
)

mqf = a2[MQF_BAND]

dnb, mqf = xr.align(
    dnb,
    mqf,
    join="inner",
)

SPATIAL_DIMS = ("y", "x")

print("A2 dimensions:", dict(a2.sizes))
print(
    "Clean DNB range:",
    float(dnb.min(skipna=True).compute()),
    "to",
    float(dnb.max(skipna=True).compute()),
)

In [ ]:
# ============================================================
# 3. LOAD GHSL G7
# ============================================================

ghsl = rxr.open_rasterio(
    GHSL_PATH,
    masked=True,
)

if "band" in ghsl.dims:
    ghsl = ghsl.isel(
        band=0,
        drop=True,
    )

if ghsl.rio.crs is None:
    raise ValueError(
        "The GHSL raster does not contain a CRS."
    )

viirs_template = dnb.isel(
    date=0,
    drop=True,
)

ghsl_viirs = ghsl.rio.reproject_match(
    viirs_template,
    resampling=Resampling.nearest,
)

ghsl_viirs = ghsl_viirs.assign_coords(
    x=viirs_template["x"],
    y=viirs_template["y"],
)

g7_mask = ghsl_viirs.isin(
    GHSL_G7_CLASSES
).fillna(False)

g7_pixel_count = int(
    g7_mask.sum().compute().item()
)

if g7_pixel_count == 0:
    raise ValueError(
        "No GHSL G7 pixels were retained after reprojection."
    )

print(
    "GHSL G7 pixels:",
    f"{g7_pixel_count:,}",
)

In [ ]:
# ============================================================
# LOAD NGCP LEYTE–SAMAR HOURLY DEMAND FROM CSV
# ============================================================

NGCP_CSV_PATH = DATA_DIR / "ngcp" / "NGCP_Hourly_Demand.csv"

if not NGCP_CSV_PATH.exists():
    raise FileNotFoundError(
        f"NGCP CSV not found:\n{NGCP_CSV_PATH}"
    )

# Rows 1–2 contain the dataset title and "Hour No." label.
# Row 3 contains the actual column headings: DATE, 1, 2, ..., 24.
ngcp_raw = pd.read_csv(
    NGCP_CSV_PATH,
    skiprows=2,
    low_memory=False,
)

ngcp_raw.columns = [
    str(column).strip()
    for column in ngcp_raw.columns
]

# Philippine dates in this file are day/month/year.
ngcp_dates = pd.to_datetime(
    ngcp_raw["DATE"].astype(str).str.strip(),
    dayfirst=True,
    errors="coerce",
).dt.normalize()

# Hour 1 = demand during the first hourly interval.
ngcp_hour_1 = pd.to_numeric(
    ngcp_raw["1"],
    errors="coerce",
)

ngcp_daily = (
    pd.DataFrame(
        {
            "date": ngcp_dates,
            "load_mw": ngcp_hour_1,
        }
    )
    .dropna(subset=["date", "load_mw"])
    .groupby("date", as_index=False)["load_mw"]
    .mean()
    .sort_values("date")
)

ngcp_load = (
    ngcp_daily
    .set_index("date")["load_mw"]
    .sort_index()
    .loc[ANALYSIS_START:ANALYSIS_END]
)

if ngcp_load.empty:
    raise ValueError(
        "The NGCP CSV was read, but there are no observations "
        f"between {ANALYSIS_START.date()} and {ANALYSIS_END.date()}."
    )

print(f"NGCP source: {NGCP_CSV_PATH}")
print(f"Available dates: {ngcp_load.index.min().date()} to "
      f"{ngcp_load.index.max().date()}")
print(f"Observations: {len(ngcp_load):,}")
print(f"Missing values: {ngcp_load.isna().sum():,}")

display(ngcp_daily.head())

## Profile A — Román-style regional recovery

The Román-style implementation uses:

- valid DNB-BRDF radiance across Samar–Leyte;
- a fixed set of pixels with positive radiance in the pre-Haiyan four-day composite;
- non-overlapping four-day mean composites;
- regional total nighttime radiance;
- recovery relative to the total radiance during 4–7 November 2013.

Four-day aggregation is performed at the pixel level before the regional radiances are summed. Pixels missing from a four-day composite contribute no observed radiance to that period. This retains the consequence of observation gaps in the original method while preventing changing spatial means from producing artificial extreme values.

In [ ]:
# ============================================================
# 5. COMMON FOUR-DAY PROFILE SETTINGS AND ROMÁN PROFILE
# ============================================================

ROMAN_BLOCK_DAYS = 4
BASELINE_DAYS = 60
SPATIAL_COMPLETENESS_PCT = 10.0
RQ_VALID_PCT = 10.0
RQ_CLIP_PERCENTILE = 95.0
MIN_BASELINE_COMPOSITES = 3

PRE_EVENT_END = EVENT_DATE - pd.Timedelta(days=1)
BASELINE_START = EVENT_DATE - pd.Timedelta(days=BASELINE_DAYS)


def build_four_day_profile(
    cube,
    fixed_mask,
    method,
    clip_percentile=None,
):
    """Create a regional four-day profile using fixed spatial support."""

    fixed_pixel_count = int(
        fixed_mask.sum().compute().item()
    )

    if fixed_pixel_count == 0:
        raise ValueError(
            f"{method}: the fixed spatial mask contains no pixels."
        )

    selected = (
        cube
        .sel(date=slice(ANALYSIS_START, PROFILE_END))
        .where(fixed_mask)
    )

    dates = pd.DatetimeIndex(
        selected["date"].values
    ).normalize()

    block_numbers = np.floor_divide(
        (dates - EVENT_DATE).days,
        ROMAN_BLOCK_DAYS,
    ).astype(int)

    selected = selected.assign_coords(
        block=("date", block_numbers)
    )

    composites = (
        selected
        .groupby("block")
        .mean(dim="date", skipna=True)
    )

    regional_signal = composites.mean(
        dim=SPATIAL_DIMS,
        skipna=True,
    )

    valid_pixel_count = composites.notnull().sum(
        dim=SPATIAL_DIMS
    )

    spatial_coverage_pct = (
        100.0
        * valid_pixel_count
        / fixed_pixel_count
    )

    profile_data = xr.Dataset(
        {
            "signal": regional_signal,
            "spatial_coverage_pct": spatial_coverage_pct,
        }
    ).compute()

    profile = (
        profile_data
        .to_dataframe()
        .reset_index()
        .sort_values("block")
        .reset_index(drop=True)
    )

    profile["date_start"] = (
        EVENT_DATE
        + pd.to_timedelta(
            profile["block"] * ROMAN_BLOCK_DAYS,
            unit="D",
        )
    )

    profile["date_end"] = (
        profile["date_start"]
        + pd.Timedelta(days=ROMAN_BLOCK_DAYS - 1)
    )

    # Apply the same minimum observation support to both profiles.
    profile.loc[
        profile["spatial_coverage_pct"]
        < SPATIAL_COMPLETENESS_PCT,
        "signal",
    ] = np.nan

    clip_upper = np.nan

    if clip_percentile is not None:
        clip_upper = profile["signal"].quantile(
            clip_percentile / 100.0
        )

        if not np.isfinite(clip_upper):
            raise ValueError(
                f"{method}: percentile clipping could not be calculated."
            )

        profile["signal"] = profile["signal"].clip(
            lower=0,
            upper=clip_upper,
        )

    baseline_mask = (
        (profile["date_start"] >= BASELINE_START)
        & (profile["date_end"] <= PRE_EVENT_END)
    )

    baseline_values = (
        profile.loc[baseline_mask, "signal"]
        .dropna()
    )

    if len(baseline_values) < MIN_BASELINE_COMPOSITES:
        raise ValueError(
            f"{method}: only {len(baseline_values)} usable "
            f"four-day composites remain in the {BASELINE_DAYS}-day "
            "pre-Haiyan baseline."
        )

    # Robust baseline across all available pre-event composites.
    baseline_reference = baseline_values.median()

    if (
        not np.isfinite(baseline_reference)
        or baseline_reference <= 0
    ):
        raise ValueError(
            f"{method}: invalid pre-Haiyan baseline "
            f"({baseline_reference})."
        )

    profile["recovery_pct"] = (
        100.0
        * profile["signal"]
        / baseline_reference
    )

    profile["method"] = method

    report = {
        "method": method,
        "fixed_pixels": fixed_pixel_count,
        "minimum_spatial_completeness_pct": (
            SPATIAL_COMPLETENESS_PCT
        ),
        "baseline_start": BASELINE_START.date(),
        "baseline_end": PRE_EVENT_END.date(),
        "baseline_composites": len(baseline_values),
        "baseline_reference": baseline_reference,
        "clip_upper": clip_upper,
    }

    return profile, report


# ------------------------------------------------------------
# Román-style input
# Basic fill-value cleaning only; no MQF or GHSL filtering.
# ------------------------------------------------------------

roman_cube = dnb.where(
    np.isfinite(dnb)
    & (dnb >= 0)
    & (dnb < 6553.5)
)

roman_baseline_cube = roman_cube.sel(
    date=slice(BASELINE_START, PRE_EVENT_END)
)

roman_baseline_observations = (
    roman_baseline_cube
    .notnull()
    .sum(dim="date")
)

roman_baseline_median = (
    roman_baseline_cube
    .median(dim="date", skipna=True)
)

roman_fixed_mask = (
    (roman_baseline_observations >= 1)
    & np.isfinite(roman_baseline_median)
    & (roman_baseline_median > 0)
)

roman_profile, roman_report = build_four_day_profile(
    cube=roman_cube,
    fixed_mask=roman_fixed_mask,
    method="Román-style whole region",
    clip_percentile=None,
)

print("Román-style profile")
print(f"Fixed pixels: {roman_report['fixed_pixels']:,}")
print(
    "Baseline composites: "
    f"{roman_report['baseline_composites']}"
)
print(
    "Baseline reference: "
    f"{roman_report['baseline_reference']:.3f}"
)

display(roman_profile.head())

In [ ]:
# ============================================================
# 6. FOUR-DAY NGCP PROFILE
# ============================================================

def four_day_series_profile(series):
    calendar = pd.date_range(
        ROMAN_BASELINE_START,
        PROFILE_END,
        freq="D",
    )

    series = series.copy()
    series.index = pd.to_datetime(
        series.index
    ).normalize()

    series = series.reindex(calendar)

    relative_days = (
        calendar - EVENT_DATE
    ).days

    block_ids = np.floor_divide(
        relative_days,
        ROMAN_BLOCK_DAYS,
    )

    frame = pd.DataFrame(
        {
            "block": block_ids,
            "value": series.values,
        }
    )

    profile = (
        frame.groupby("block", as_index=False)
        .agg(
            value=("value", "mean"),
            observed_days=("value", "count"),
        )
    )

    profile = (
        profile.set_index("block")
        .reindex(np.arange(-1, 45))
        .rename_axis("block")
        .reset_index()
    )

    baseline = profile.loc[
        profile["block"] == -1,
        "value",
    ].iloc[0]

    profile["recovery_pct"] = (
        100.0
        * profile["value"]
        / baseline
    )

    profile["date_start"] = (
        EVENT_DATE
        + pd.to_timedelta(
            profile["block"]
            * ROMAN_BLOCK_DAYS,
            unit="D",
        )
    )

    profile["date_end"] = (
        profile["date_start"]
        + pd.Timedelta(days=3)
    )

    return profile


ngcp_roman_profile = four_day_series_profile(
    ngcp_load
)

display(
    ngcp_roman_profile.head().round(2)
)

## Profile B — Reliability-qualified regional recovery

The reliability-qualified implementation follows the established Samar–Leyte processing:

- direct DNB-BRDF rather than gap-filled radiance;
- `MQF == 0`;
- fixed GHSL G7 settlement support;
- daily spatial-completeness threshold of 60%;
- lower clipping at zero and upper clipping at the 95th percentile;
- centred 30-day moving average;
- standardisation using the 180-day pre-Haiyan period.

NGCP Hour 1 load is processed with the same centred moving window and pre-event standardisation. Daily spatial completeness is retained as an explicit diagnostic.

In [ ]:
# ============================================================
# 7. RELIABILITY-QUALIFIED FOUR-DAY PROFILE
# ============================================================

# MQF == 0 observations within GHSL G7.
rq_cube = roman_cube.where(
    (mqf == 0) & g7_mask
)

# Fixed GHSL G7 support is defined using the same 60-day baseline.
rq_baseline_cube = rq_cube.sel(
    date=slice(BASELINE_START, PRE_EVENT_END)
)

rq_baseline_observations = (
    rq_baseline_cube
    .notnull()
    .sum(dim="date")
)

rq_baseline_median = (
    rq_baseline_cube
    .median(dim="date", skipna=True)
)

rq_fixed_mask = (
    g7_mask
    & (rq_baseline_observations >= 1)
    & np.isfinite(rq_baseline_median)
    & (rq_baseline_median > 0)
)

rq_profile, rq_report = build_four_day_profile(
    cube=rq_cube,
    fixed_mask=rq_fixed_mask,
    method="Reliability-qualified GHSL G7",
    clip_percentile=RQ_CLIP_PERCENTILE,
)

print("Reliability-qualified profile")
print(f"Fixed pixels: {rq_report['fixed_pixels']:,}")
print(
    "Baseline composites: "
    f"{rq_report['baseline_composites']}"
)
print(
    "Baseline reference: "
    f"{rq_report['baseline_reference']:.3f}"
)
print(
    f"{RQ_CLIP_PERCENTILE:.0f}th-percentile limit: "
    f"{rq_report['clip_upper']:.3f}"
)

display(rq_profile.head())

In [ ]:
# ============================================================
# 8. NGCP FOUR-DAY PROFILE AND LIKE-FOR-LIKE PLOT
# ============================================================

ngcp_series = (
    ngcp_load
    .loc[ANALYSIS_START:PROFILE_END]
    .dropna()
    .astype(float)
)

ngcp_blocks = np.floor_divide(
    (ngcp_series.index - EVENT_DATE).days,
    ROMAN_BLOCK_DAYS,
).astype(int)

ngcp_composites = (
    ngcp_series
    .groupby(ngcp_blocks)
    .mean()
)

ngcp_profile = pd.DataFrame(
    {
        "block": ngcp_composites.index.astype(int),
        "load_mw": ngcp_composites.values,
    }
)

ngcp_profile["date_start"] = (
    EVENT_DATE
    + pd.to_timedelta(
        ngcp_profile["block"] * ROMAN_BLOCK_DAYS,
        unit="D",
    )
)

ngcp_profile["date_end"] = (
    ngcp_profile["date_start"]
    + pd.Timedelta(days=ROMAN_BLOCK_DAYS - 1)
)

ngcp_baseline_mask = (
    (ngcp_profile["date_start"] >= BASELINE_START)
    & (ngcp_profile["date_end"] <= PRE_EVENT_END)
)

ngcp_baseline_values = (
    ngcp_profile.loc[
        ngcp_baseline_mask,
        "load_mw",
    ]
    .dropna()
)

if len(ngcp_baseline_values) < MIN_BASELINE_COMPOSITES:
    raise ValueError(
        "Insufficient NGCP four-day composites in the "
        "60-day pre-Haiyan baseline."
    )

ngcp_baseline_reference = (
    ngcp_baseline_values.median()
)

ngcp_profile["recovery_pct"] = (
    100.0
    * ngcp_profile["load_mw"]
    / ngcp_baseline_reference
)

print(
    "NGCP baseline reference: "
    f"{ngcp_baseline_reference:.3f} MW "
    f"from {len(ngcp_baseline_values)} composites"
)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=(
        "a. Román-style whole-region profile",
        "b. Reliability-qualified GHSL G7 profile",
    ),
)

fig.add_trace(
    go.Scatter(
        x=roman_profile["date_start"],
        y=roman_profile["recovery_pct"],
        mode="lines+markers",
        name="Román-style NTL",
        connectgaps=False,
        line=dict(
            color="#D55E00",
            width=2.5,
            shape="hv",
        ),
        marker=dict(size=5),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=ngcp_profile["date_start"],
        y=ngcp_profile["recovery_pct"],
        mode="lines+markers",
        name="NGCP 1 AM",
        connectgaps=False,
        line=dict(
            color="#26364A",
            width=2.5,
            shape="hv",
        ),
        marker=dict(size=4),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=rq_profile["date_start"],
        y=rq_profile["recovery_pct"],
        mode="lines+markers",
        name="Reliability-qualified NTL",
        connectgaps=False,
        line=dict(
            color="#009E73",
            width=2.5,
            shape="hv",
        ),
        marker=dict(size=5),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=ngcp_profile["date_start"],
        y=ngcp_profile["recovery_pct"],
        mode="lines+markers",
        name="NGCP 1 AM",
        showlegend=False,
        connectgaps=False,
        line=dict(
            color="#26364A",
            width=2.5,
            shape="hv",
        ),
        marker=dict(size=4),
    ),
    row=2,
    col=1,
)

for row_number in (1, 2):
    fig.add_hline(
        y=100,
        line=dict(
            color="#7F8C8D",
            width=1.5,
            dash="dot",
        ),
        row=row_number,
        col=1,
    )

    fig.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(
            color="#0057FF",
            width=2,
            dash="dash",
        ),
        row=row_number,
        col=1,
    )

    for boundary_day in (60, 120):
        fig.add_vline(
            x=(
                EVENT_DATE
                + pd.Timedelta(days=boundary_day)
            ).to_pydatetime(),
            line=dict(
                color="#A8B6CC",
                width=1.5,
                dash="dot",
            ),
            row=row_number,
            col=1,
        )

fig.add_annotation(
    x=EVENT_DATE,
    y=0.91,
    xref="x",
    yref="paper",
    text="Haiyan",
    showarrow=False,
    xanchor="left",
    font=dict(
        color="#0057FF",
        size=14,
    ),
)

fig.update_yaxes(
    title_text="Output relative to baseline (%)",
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Output relative to baseline (%)",
    row=2,
    col=1,
)

fig.update_xaxes(
    title_text="Four-day composite period",
    range=[ANALYSIS_START, PROFILE_END],
    row=2,
    col=1,
)

fig.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1200,
    height=900,
    legend=dict(
        orientation="h",
        yanchor="middle",
        y=0.5,
        xanchor="center",
        x=0.8,
        font=dict(size=14),
    ),
    font=dict(
        family="Arial",
        size=15,
        color="#243B5A",
    ),
    margin=dict(
        l=90,
        r=40,
        t=150,
        b=70,
    ),
    hovermode="x unified",
)

fig.show()

In [ ]:
# ============================================================
# 9. BASELINE AND NTL–NGCP ALIGNMENT SUMMARY
# ============================================================

baseline_summary = pd.DataFrame(
    [
        roman_report,
        rq_report,
        {
            "method": "NGCP 1 AM",
            "fixed_pixels": np.nan,
            "minimum_spatial_completeness_pct": np.nan,
            "baseline_start": BASELINE_START.date(),
            "baseline_end": PRE_EVENT_END.date(),
            "baseline_composites": len(
                ngcp_baseline_values
            ),
            "baseline_reference": (
                ngcp_baseline_reference
            ),
            "clip_upper": np.nan,
        },
    ]
)

display(baseline_summary)


def calculate_alignment(
    ntl_profile,
    method,
):
    paired = (
        ntl_profile[
            [
                "block",
                "recovery_pct",
            ]
        ]
        .rename(
            columns={
                "recovery_pct": "ntl",
            }
        )
        .merge(
            ngcp_profile[
                [
                    "block",
                    "recovery_pct",
                ]
            ].rename(
                columns={
                    "recovery_pct": "ngcp",
                }
            ),
            on="block",
            how="inner",
        )
    )

    paired = (
        paired
        .loc[paired["block"] >= 0]
        .dropna(
            subset=[
                "ntl",
                "ngcp",
            ]
        )
    )

    if len(paired) < 2:
        return {
            "method": method,
            "n": len(paired),
            "pearson_r": np.nan,
            "spearman_r": np.nan,
            "rmse": np.nan,
            "mae": np.nan,
        }

    difference = paired["ntl"] - paired["ngcp"]

    return {
        "method": method,
        "n": len(paired),
        "pearson_r": paired["ntl"].corr(
            paired["ngcp"],
            method="pearson",
        ),
        "spearman_r": paired["ntl"].corr(
            paired["ngcp"],
            method="spearman",
        ),
        "rmse": np.sqrt(
            np.mean(np.square(difference))
        ),
        "mae": np.mean(np.abs(difference)),
    }


alignment_summary = pd.DataFrame(
    [
        calculate_alignment(
            roman_profile,
            "Román-style whole region",
        ),
        calculate_alignment(
            rq_profile,
            "Reliability-qualified GHSL G7",
        ),
    ]
)

metric_columns = [
    "pearson_r",
    "spearman_r",
    "rmse",
    "mae",
]

alignment_summary[metric_columns] = (
    alignment_summary[metric_columns]
    .round(3)
)

display(alignment_summary)

In [ ]:
# ============================================================
# 5. ROMÁN-STYLE PIXEL-MATCHED FOUR-DAY PROFILE
# ============================================================

ROMAN_BLOCK_DAYS = 4
BASELINE_DAYS = 60
SPATIAL_COMPLETENESS_PCT = 10.0
RQ_CLIP_PERCENTILE = 95.0

PRE_EVENT_END = EVENT_DATE - pd.Timedelta(days=1)
BASELINE_START = EVENT_DATE - pd.Timedelta(days=BASELINE_DAYS)

STAGE_WINDOWS = {
    "Baseline": (
        BASELINE_START,
        PRE_EVENT_END,
    ),
    "Stage 1 (0–59 days)": (
        EVENT_DATE,
        EVENT_DATE + pd.Timedelta(days=59),
    ),
    "Stage 2 (60–119 days)": (
        EVENT_DATE + pd.Timedelta(days=60),
        EVENT_DATE + pd.Timedelta(days=119),
    ),
    "Stage 3 (120–179 days)": (
        EVENT_DATE + pd.Timedelta(days=120),
        EVENT_DATE + pd.Timedelta(days=179),
    ),
}


def clean_ntl_values(data_array):
    """Remove known VNP46 radiance fill values."""

    cleaned = data_array.astype("float32")

    for fill_value in (
        -9999.0,
        -32768.0,
        6553.5,
        65535.0,
    ):
        cleaned = cleaned.where(
            cleaned != fill_value
        )

    return cleaned.where(
        np.isfinite(cleaned)
        & (cleaned >= 0)
    )


def build_pixel_matched_profile(
    cube,
    base_mask,
    method,
    input_band,
):
    """
    Calculate four-day NTL recovery using a per-pixel baseline.

    For each composite, the numerator and denominator use exactly
    the same valid pixels:

        Recovery = 100 × Σ(NTL_i) / Σ(NTL_0)
    """

    selected = (
        cube
        .sel(date=slice(ANALYSIS_START, PROFILE_END))
        .where(base_mask)
    )

    dates = pd.DatetimeIndex(
        selected["date"].values
    ).normalize()

    block_numbers = np.floor_divide(
        (dates - EVENT_DATE).days,
        ROMAN_BLOCK_DAYS,
    ).astype(int)

    selected = selected.assign_coords(
        block=("date", block_numbers)
    )

    # Non-overlapping four-day mean composites.
    composites = (
        selected
        .groupby("block")
        .mean(dim="date", skipna=True)
    )

    composite_blocks = (
        composites["block"]
        .values
        .astype(int)
    )

    composite_start = (
        EVENT_DATE
        + pd.to_timedelta(
            composite_blocks * ROMAN_BLOCK_DAYS,
            unit="D",
        )
    )

    composite_end = (
        composite_start
        + pd.Timedelta(days=ROMAN_BLOCK_DAYS - 1)
    )

    baseline_blocks = composite_blocks[
        (composite_start >= BASELINE_START)
        & (composite_end <= PRE_EVENT_END)
    ]

    if len(baseline_blocks) == 0:
        raise ValueError(
            f"{method}: no four-day baseline composites were found."
        )

    # Per-pixel NTL0: median of pre-Haiyan four-day composites.
    baseline_composites = composites.sel(
        block=baseline_blocks
    )

    baseline_observations = (
        baseline_composites
        .notnull()
        .sum(dim="block")
    )

    ntl0 = (
        baseline_composites
        .median(dim="block", skipna=True)
        .compute()
    )

    fixed_mask = (
        base_mask
        & (baseline_observations >= 1)
        & np.isfinite(ntl0)
        & (ntl0 > 0)
    ).compute()

    fixed_pixel_count = int(
        fixed_mask.sum().item()
    )

    if fixed_pixel_count == 0:
        raise ValueError(
            f"{method}: no valid baseline-lit pixels were found."
        )

    # Use matching pixels in NTL_i and NTL_0.
    paired_valid = (
        composites.notnull()
        & fixed_mask
        & ntl0.notnull()
    )

    valid_pixel_count = paired_valid.sum(
        dim=SPATIAL_DIMS
    )

    spatial_coverage_pct = (
        100.0
        * valid_pixel_count
        / fixed_pixel_count
    )

    current_radiance = (
        composites
        .where(paired_valid)
        .sum(
            dim=SPATIAL_DIMS,
            skipna=True,
            min_count=1,
        )
    )

    matched_baseline_radiance = (
        ntl0
        .where(paired_valid)
        .sum(
            dim=SPATIAL_DIMS,
            skipna=True,
            min_count=1,
        )
    )

    recovery_pct = (
        100.0
        * current_radiance
        / matched_baseline_radiance
    )

    reduced = xr.Dataset(
        {
            "current_radiance": current_radiance,
            "matched_baseline_radiance": (
                matched_baseline_radiance
            ),
            "recovery_pct": recovery_pct,
            "spatial_coverage_pct": (
                spatial_coverage_pct
            ),
            "valid_pixel_count": valid_pixel_count,
        }
    ).compute()

    profile = (
        reduced
        .to_dataframe()
        .reset_index()
        .sort_values("block")
        .reset_index(drop=True)
    )

    profile["date_start"] = (
        EVENT_DATE
        + pd.to_timedelta(
            profile["block"] * ROMAN_BLOCK_DAYS,
            unit="D",
        )
    )

    profile["date_end"] = (
        profile["date_start"]
        + pd.Timedelta(days=ROMAN_BLOCK_DAYS - 1)
    )

    # Do not interpret composites below 10% spatial support.
    profile.loc[
        profile["spatial_coverage_pct"]
        < SPATIAL_COMPLETENESS_PCT,
        [
            "current_radiance",
            "matched_baseline_radiance",
            "recovery_pct",
        ],
    ] = np.nan

    profile["method"] = method

    baseline_profile_mask = (
        (profile["date_start"] >= BASELINE_START)
        & (profile["date_end"] <= PRE_EVENT_END)
    )

    report = {
        "method": method,
        "input_band": input_band,
        "baseline_start": BASELINE_START.date(),
        "baseline_end": PRE_EVENT_END.date(),
        "baseline_composites": len(baseline_blocks),
        "fixed_pixels": fixed_pixel_count,
        "minimum_completeness_pct": (
            SPATIAL_COMPLETENESS_PCT
        ),
        "median_baseline_coverage_pct": (
            profile.loc[
                baseline_profile_mask,
                "spatial_coverage_pct",
            ].median()
        ),
        "median_pixel_ntl0": float(
            ntl0
            .where(fixed_mask)
            .median(
                dim=SPATIAL_DIMS,
                skipna=True,
            )
            .item()
        ),
    }

    return (
        profile,
        composites.where(fixed_mask),
        ntl0.where(fixed_mask),
        fixed_mask,
        report,
    )


# ------------------------------------------------------------
# Román-style input: directly observed DNB-BRDF
# ------------------------------------------------------------

roman_cube = clean_ntl_values(dnb)

# No GHSL or MQF restriction in the Román-style branch.
roman_base_mask = xr.ones_like(
    roman_cube.isel(date=0),
    dtype=bool,
)

(
    roman_profile,
    roman_composites,
    roman_ntl0,
    roman_fixed_mask,
    roman_report,
) = build_pixel_matched_profile(
    cube=roman_cube,
    base_mask=roman_base_mask,
    method="Román-style transfer",
    input_band=(
        "Direct DNB_BRDF_Corrected_NTL; "
        "basic fill-value screening"
    ),
)

print("Román-style pixel-matched profile")
print(f"Input: {roman_report['input_band']}")
print(
    f"Baseline: {BASELINE_START.date()} to "
    f"{PRE_EVENT_END.date()}"
)
print(
    "Baseline composites: "
    f"{roman_report['baseline_composites']}"
)
print(
    "Baseline-lit pixels: "
    f"{roman_report['fixed_pixels']:,}"
)
print(
    "Median baseline coverage: "
    f"{roman_report['median_baseline_coverage_pct']:.1f}%"
)

display(roman_profile.head())

In [ ]:
# ============================================================
# 7. RELIABILITY-QUALIFIED PIXEL-MATCHED PROFILE
# ============================================================

direct_dnb = clean_ntl_values(dnb)

# Direct observations satisfying MQF == 0 and GHSL G7.
rq_unclipped = direct_dnb.where(
    (mqf == 0)
    & g7_mask
)

# Apply the RQ1-style daily spatial 95th-percentile clamp.
rq_quantile_source = rq_unclipped.chunk(
    {
        dimension: -1
        for dimension in SPATIAL_DIMS
    }
)

rq_daily_p95 = (
    rq_quantile_source
    .quantile(
        RQ_CLIP_PERCENTILE / 100.0,
        dim=SPATIAL_DIMS,
        skipna=True,
    )
    .squeeze(drop=True)
    .compute()
)

rq_cube = xr.where(
    rq_unclipped > rq_daily_p95,
    rq_daily_p95,
    rq_unclipped,
)

(
    rq_profile,
    rq_composites,
    rq_ntl0,
    rq_fixed_mask,
    rq_report,
) = build_pixel_matched_profile(
    cube=rq_cube,
    base_mask=g7_mask,
    method="Reliability-qualified GHSL G7",
    input_band=(
        "Direct DNB-BRDF; MQF == 0; "
        "GHSL G7; daily P95 clamp"
    ),
)

print("Reliability-qualified pixel-matched profile")
print(f"Input: {rq_report['input_band']}")
print(
    f"Baseline: {BASELINE_START.date()} to "
    f"{PRE_EVENT_END.date()}"
)
print(
    "Baseline composites: "
    f"{rq_report['baseline_composites']}"
)
print(
    f"Baseline-lit G7 pixels: "
    f"{rq_report['fixed_pixels']:,}"
)
print(
    "Median baseline coverage: "
    f"{rq_report['median_baseline_coverage_pct']:.1f}%"
)
print(
    "Minimum retained spatial completeness: "
    f"{SPATIAL_COMPLETENESS_PCT:.0f}%"
)

display(rq_profile.head())

In [ ]:
# ============================================================
# 8. NGCP FOUR-DAY PROFILE AND LIKE-FOR-LIKE PLOT
# ============================================================

ngcp_series = (
    ngcp_load
    .loc[ANALYSIS_START:PROFILE_END]
    .dropna()
    .astype(float)
)

ngcp_blocks = np.floor_divide(
    (ngcp_series.index - EVENT_DATE).days,
    ROMAN_BLOCK_DAYS,
).astype(int)

ngcp_composites = (
    ngcp_series
    .groupby(ngcp_blocks)
    .mean()
)

ngcp_profile = pd.DataFrame(
    {
        "block": ngcp_composites.index.astype(int),
        "load_mw": ngcp_composites.values,
    }
)

ngcp_profile["date_start"] = (
    EVENT_DATE
    + pd.to_timedelta(
        ngcp_profile["block"] * ROMAN_BLOCK_DAYS,
        unit="D",
    )
)

ngcp_profile["date_end"] = (
    ngcp_profile["date_start"]
    + pd.Timedelta(days=ROMAN_BLOCK_DAYS - 1)
)

ngcp_baseline_mask = (
    (ngcp_profile["date_start"] >= BASELINE_START)
    & (ngcp_profile["date_end"] <= PRE_EVENT_END)
)

ngcp_baseline_values = (
    ngcp_profile.loc[
        ngcp_baseline_mask,
        "load_mw",
    ]
    .dropna()
)

if len(ngcp_baseline_values) < MIN_BASELINE_COMPOSITES:
    raise ValueError(
        "Insufficient NGCP four-day composites in the "
        "60-day pre-Haiyan baseline."
    )

ngcp_baseline_reference = (
    ngcp_baseline_values.median()
)

ngcp_profile["recovery_pct"] = (
    100.0
    * ngcp_profile["load_mw"]
    / ngcp_baseline_reference
)

print(
    "NGCP baseline reference: "
    f"{ngcp_baseline_reference:.3f} MW "
    f"from {len(ngcp_baseline_values)} composites"
)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=(
        "a. Román-style whole-region profile",
        "b. Reliability-qualified GHSL G7 profile",
    ),
)

fig.add_trace(
    go.Scatter(
        x=roman_profile["date_start"],
        y=roman_profile["recovery_pct"],
        mode="lines+markers",
        name="Román-style NTL",
        connectgaps=False,
        line=dict(
            color="#D55E00",
            width=2.5,
            shape="hv",
        ),
        marker=dict(size=5),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=ngcp_profile["date_start"],
        y=ngcp_profile["recovery_pct"],
        mode="lines+markers",
        name="NGCP 1 AM",
        connectgaps=False,
        line=dict(
            color="#26364A",
            width=2.5,
            shape="hv",
        ),
        marker=dict(size=4),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=rq_profile["date_start"],
        y=rq_profile["recovery_pct"],
        mode="lines+markers",
        name="Reliability-qualified NTL",
        connectgaps=False,
        line=dict(
            color="#009E73",
            width=2.5,
            shape="hv",
        ),
        marker=dict(size=5),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=ngcp_profile["date_start"],
        y=ngcp_profile["recovery_pct"],
        mode="lines+markers",
        name="NGCP 1 AM",
        showlegend=False,
        connectgaps=False,
        line=dict(
            color="#26364A",
            width=2.5,
            shape="hv",
        ),
        marker=dict(size=4),
    ),
    row=2,
    col=1,
)

for row_number in (1, 2):
    fig.add_hline(
        y=100,
        line=dict(
            color="#7F8C8D",
            width=1.5,
            dash="dot",
        ),
        row=row_number,
        col=1,
    )

    fig.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(
            color="#0057FF",
            width=2,
            dash="dash",
        ),
        row=row_number,
        col=1,
    )

    for boundary_day in (60, 120):
        fig.add_vline(
            x=(
                EVENT_DATE
                + pd.Timedelta(days=boundary_day)
            ).to_pydatetime(),
            line=dict(
                color="#A8B6CC",
                width=1.5,
                dash="dot",
            ),
            row=row_number,
            col=1,
        )

fig.add_annotation(
    x=EVENT_DATE,
    y=0.91,
    xref="x",
    yref="paper",
    text="Haiyan",
    showarrow=False,
    xanchor="left",
    font=dict(
        color="#0057FF",
        size=14,
    ),
)

fig.update_yaxes(
    title_text="Output relative to baseline (%)",
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Output relative to baseline (%)",
    row=2,
    col=1,
)

fig.update_xaxes(
    title_text="Four-day composite period",
    range=[ANALYSIS_START, PROFILE_END],
    row=2,
    col=1,
)

fig.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1200,
    height=900,
    legend=dict(
        orientation="h",
        yanchor="middle",
        y=0.5,
        xanchor="center",
        x=0.8,
        font=dict(size=14),
    ),
    font=dict(
        family="Arial",
        size=15,
        color="#243B5A",
    ),
    margin=dict(
        l=90,
        r=40,
        t=150,
        b=70,
    ),
    hovermode="x unified",
)

fig.show()

In [ ]:
# ============================================================
# 8B. PREPARE DAILY AND FOUR-DAY DIAGNOSTIC SERIES
# ============================================================

# ------------------------------------------------------------
# Load gap-filled NTL for the raw diagnostic only
# ------------------------------------------------------------

gap_dataset = xr.open_zarr(
    A2_ZARR_PATH,
    consolidated=None,
)

if (
    "processed" in gap_dataset.dims
    and "date" in gap_dataset.coords
):
    gap_dataset = gap_dataset.swap_dims(
        {"processed": "date"}
    )

gap_filled_cube = clean_ntl_values(
    gap_dataset[
        "Gap_Filled_DNB_BRDF_Corrected_NTL"
    ]
)

gap_filled_cube = gap_filled_cube.assign_coords(
    date=pd.to_datetime(
        gap_filled_cube["date"].values
    )
).sortby("date")


# ------------------------------------------------------------
# Extract daily regional mean NTL and spatial completeness
# ------------------------------------------------------------

def extract_daily_ntl_and_sc(
    cube,
    fixed_mask,
):
    """Daily regional mean and spatial completeness."""

    fixed_mask = (
        fixed_mask
        .fillna(False)
        .astype(bool)
    )

    fixed_pixel_count = int(
        fixed_mask.sum().compute().item()
    )

    if fixed_pixel_count == 0:
        raise ValueError(
            "The fixed mask contains no pixels."
        )

    selected = (
        cube
        .sel(date=slice(ANALYSIS_START, PROFILE_END))
        .where(fixed_mask)
    )

    valid_pixel_count = selected.notnull().sum(
        dim=SPATIAL_DIMS
    )

    daily_mean_ntl = selected.mean(
        dim=SPATIAL_DIMS,
        skipna=True,
    )

    daily_sc = (
        100.0
        * valid_pixel_count
        / fixed_pixel_count
    )

    daily_data = xr.Dataset(
        {
            "mean_ntl": daily_mean_ntl,
            "sc_pct": daily_sc,
            "valid_pixel_count": valid_pixel_count,
        }
    ).compute()

    daily_frame = (
        daily_data
        .to_dataframe()
        .reset_index()
    )

    daily_frame["date"] = pd.to_datetime(
        daily_frame["date"]
    ).dt.normalize()

    daily_frame = (
        daily_frame
        .groupby("date", as_index=False)
        .mean(numeric_only=True)
        .sort_values("date")
        .set_index("date")
    )

    return daily_frame


# Direct daily DNB using Román spatial support.
roman_daily = extract_daily_ntl_and_sc(
    cube=roman_cube,
    fixed_mask=roman_fixed_mask,
)

# Gap-filled daily NTL using the same Román support.
gap_filled_daily = extract_daily_ntl_and_sc(
    cube=gap_filled_cube,
    fixed_mask=roman_fixed_mask,
)

# Reliability-qualified daily NTL using G7 support.
rq_daily = extract_daily_ntl_and_sc(
    cube=rq_cube,
    fixed_mask=rq_fixed_mask,
)


# ------------------------------------------------------------
# Raw four-day mean radiance before baseline normalization
# ------------------------------------------------------------

roman_profile["raw_mean_ntl"] = (
    roman_profile["current_radiance"]
    / roman_profile["valid_pixel_count"]
)

rq_profile["raw_mean_ntl"] = (
    rq_profile["current_radiance"]
    / rq_profile["valid_pixel_count"]
)


In [ ]:
rq_profile

In [ ]:


# ------------------------------------------------------------
# Shared plotting helper
# ------------------------------------------------------------

SC_COLORSCALE = [
    [0.00, "#F7FCF5"],
    [0.10, "#E5F5E0"],
    [0.25, "#C7E9C0"],
    [0.50, "#74C476"],
    [0.75, "#238B45"],
    [1.00, "#005A32"],
]

DNB_COLOR = "#0091FF"
DNB_FOUR_DAY_COLOR = "#002FFF"
GAP_FILLED_COLOR = "#FF0000"
RQ_COLOR = "#00C54F"
RQ_FOUR_DAY_COLOR = "#009227"
NGCP_COLOR = "#000000"


def plot_sc_timeseries(
    title,
    ntl_panel_title,
    sc_series,
    line_series,
    y_axis_title,
):
    """
    Plot spatial-completeness strips above a time series.

    sc_series: list of dictionaries containing label, x and y.
    line_series: list of dictionaries containing the line settings.
    """

    figure = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=[
            0.10,
            0.85,
        ],
        subplot_titles=(
            "Spatial Completeness (SC)",
            ntl_panel_title,
        ),
    )

    # Spatial-completeness strips
    # Align all SC series to a shared date axis before plotting.

    sc_dates = pd.DatetimeIndex(
        sorted(
            set().union(
                *[
                    pd.to_datetime(
                        sc_item["x"]
                    ).tolist()
                    for sc_item in sc_series
                ]
            )
        )
    )

    sc_labels = []
    sc_matrix = []

    for sc_item in sc_series:
        sc_frame = pd.DataFrame(
            {
                "date": pd.to_datetime(
                    sc_item["x"]
                ),
                "sc_pct": np.asarray(
                    sc_item["y"],
                    dtype=float,
                ),
            }
        )

        sc_frame = (
            sc_frame
            .groupby("date", as_index=True)["sc_pct"]
            .mean()
            .reindex(sc_dates)
        )

        sc_labels.append(
            sc_item["label"]
        )

        sc_matrix.append(
            sc_frame.to_numpy(
                dtype=float
            )
        )

    figure.add_trace(
        go.Heatmap(
            x=sc_dates,
            y=sc_labels,
            z=np.vstack(sc_matrix),
            coloraxis="coloraxis",
            zsmooth=False,
            hoverongaps=False,
            hovertemplate=(
                "%{x|%d %b %Y}<br>"
                "Support: %{y}<br>"
                "Spatial completeness: %{z:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

    # Time-series lines
    for line_item in line_series:
        figure.add_trace(
            go.Scatter(
                x=pd.to_datetime(line_item["x"]),
                y=line_item["y"],
                mode=line_item.get(
                    "mode",
                    "lines",
                ),
                name=line_item["name"],
                connectgaps=False,
                opacity=line_item.get(
                    "opacity",
                    1.0,
                ),
                line=dict(
                    color=line_item["color"],
                    width=line_item.get(
                        "width",
                        2.0,
                    ),
                    dash=line_item.get(
                        "dash",
                        "solid",
                    ),
                    shape=line_item.get(
                        "shape",
                        "linear",
                    ),
                ),
                marker=dict(
                    size=line_item.get(
                        "marker_size",
                        4,
                    ),
                ),
                hovertemplate=(
                    "%{x|%d %b %Y}<br>"
                    f"{line_item['name']}: "
                    "%{y:.2f} "
                    f"{line_item['unit']}"
                    "<extra></extra>"
                ),
            ),
            row=2,
            col=1,
        )

    # Haiyan and fixed 60-day intervals
    for row_number in (1, 2):
        figure.add_vline(
            x=EVENT_DATE.to_pydatetime(),
            line=dict(
                color="#0057FF",
                width=2,
                dash="dash",
            ),
            row=row_number,
            col=1,
        )

        for boundary_day in (60, 120):
            figure.add_vline(
                x=(
                    EVENT_DATE
                    + pd.Timedelta(
                        days=boundary_day
                    )
                ).to_pydatetime(),
                line=dict(
                    color="#A8B6CC",
                    width=1.5,
                    dash="dot",
                ),
                row=row_number,
                col=1,
            )

    figure.add_annotation(
        x=(EVENT_DATE + pd.Timedelta(days=5)).to_pydatetime(),
        y=0.05,
        xref="x2",
        yref="y2 domain",
        text="Haiyan",
        showarrow=False,
        xanchor="left",
        font=dict(
            color="#0057FF",
            size=16,
        ),
    )

    sc_labels = [
        item["label"]
        for item in sc_series
    ]

    figure.update_yaxes(
        categoryorder="array",
        categoryarray=sc_labels[::-1],
        row=1,
        col=1,
    )

    figure.update_yaxes(
        title_text=y_axis_title,
        row=2,
        col=1,
    )

    figure.update_xaxes(
        range=[
            ANALYSIS_START,
            PROFILE_END,
        ],
    )

    figure.update_xaxes(
        title_text="Date",
        row=2,
        col=1,
    )

    figure.update_layout(
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        width=1200,
        height=600,
        title=dict(
            text=title,
            x=0.5,
            y=0.92,
            xanchor="center",
            font=dict(size=24),
        ),
        legend=dict(
            orientation="h",
            yanchor="middle",
            y=0.8,
            xanchor="center",
            x=0.8,
            font=dict(size=16),
        ),
        coloraxis=dict(
            colorscale=SC_COLORSCALE,
            cmin=0,
            cmax=100,
            colorbar=dict(
                title=dict(
                    text="SC (%)"
                ),
                x=1.015,
                xanchor="left",
                y=0.98,
                len=0.21,
                thickness=18,
                outlinewidth=0.8,
                outlinecolor="#555555",
            ),
        ),
        font=dict(
            family="Arial",
            size=16,
            color="#243B5A",
        ),
        margin=dict(
            l=105,
            r=105,
            t=140,
            b=70,
        ),
        hovermode="x",
    )

    return figure

In [ ]:
# ------------------------------------------------------------
# ROMÁN 1:
# Raw daily direct DNB-BRDF versus daily gap-filled NTL
# ------------------------------------------------------------

fig_roman_raw = plot_sc_timeseries(
    title=(""
        # "Daily raw and gap-filled DNB-BRDF NTL"
    ),
    ntl_panel_title=(""
        # "Daily Regional Mean NTL Radiance"
    ),
    sc_series=[
        {
            "label": "",
            "x": roman_daily.index,
            "y": roman_daily["sc_pct"],
        },
    ],
    line_series=[
        {
            "name": "DNB-BRDF",
            "x": roman_daily.index,
            "y": roman_daily["mean_ntl"],
            "color": DNB_COLOR,
            "width": 1.5,
            "mode": "lines+markers",
            "marker_size": 3,
            "opacity": 0.80,
            "unit": "nW cm⁻² sr⁻¹",
        },
        {
            "name": "Gap-filled DNB-BRDF",
            "x": gap_filled_daily.index,
            "y": gap_filled_daily["mean_ntl"],
            "color": GAP_FILLED_COLOR,
            "width": 2.0,
            "mode": "lines",
            "unit": "nW cm⁻² sr⁻¹",
        },
    ],
    y_axis_title=(
        "Mean DNB-BRDF<br>"
        "(nW cm⁻² sr⁻¹)"
    ),
)

fig_roman_raw.update_yaxes(type='log',row=2, col=1)
fig_roman_raw.show()

In [ ]:
# ------------------------------------------------------------
# ROMÁN 2:
# Raw daily DNB-BRDF versus four-day aggregation
# ------------------------------------------------------------

fig_roman_aggregation = plot_sc_timeseries(
    title="",#"Daily BRDF to Four-Day Aggregation",
    ntl_panel_title=(""
        # "Raw daily DNB-BRDF versus four-day mean"
    ),
    sc_series=[
        {
            "label": "",
            "x": roman_profile["date_start"],
            "y": roman_profile["spatial_coverage_pct"],
        },
    ],
    line_series=[
        {
            "name": "Raw daily DNB-BRDF",
            "x": roman_daily.index,
            "y": roman_daily["mean_ntl"],
            "color": DNB_COLOR,
            "width": 1.3,
            "mode": "lines+markers",
            "marker_size": 3,
            "opacity": 0.60,
            "unit": "nW cm⁻² sr⁻¹",
        },
        {
            "name": "Four-day mean DNB-BRDF",
            "x": roman_profile["date_start"],
            "y": roman_profile["raw_mean_ntl"],
            "color": DNB_FOUR_DAY_COLOR,
            "width": 2.8,
            "mode": "lines+markers",
            "marker_size": 5,
            "shape": "hv",
            "unit": "nW cm⁻² sr⁻¹",
        },
    ],
    y_axis_title=(
        "Mean DNB-BRDF<br>"
        "(nW cm⁻² sr⁻¹)"
    ),
)

fig_roman_aggregation.update_yaxes(type='log',row=2, col=1)
fig_roman_aggregation.show()

In [ ]:
# ------------------------------------------------------------
# ROMÁN 3:
# Baseline-normalized four-day DNB versus four-day NGCP
# ------------------------------------------------------------

fig_roman_normalized = plot_sc_timeseries(
    title=(""
        # "Baseline-normalized "
        # "NTL and NGCP"
    ),
    ntl_panel_title=(""
        # "Four-day output relative to pre-Haiyan baseline"
    ),
    sc_series=[
        {
            "label": "",
            "x": roman_profile["date_start"],
            "y": roman_profile[
                "spatial_coverage_pct"
            ],
        },
    ],
    line_series=[
        {
            "name": "DNB-BRDF (4D)",
            "x": roman_profile["date_start"],
            "y": roman_profile["recovery_pct"],
            "color": DNB_FOUR_DAY_COLOR,
            "width": 2.8,
            "mode": "lines+markers",
            "marker_size": 5,
            "shape": "hv",
            "unit": "%",
        },
        {
            "name": "NGCP 1 AM (4D)",
            "x": ngcp_profile["date_start"],
            "y": ngcp_profile["recovery_pct"],
            "color": NGCP_COLOR,
            "width": 2.5,
            "mode": "lines+markers",
            "marker_size": 4,
            "shape": "hv",
            "unit": "%",
        },
    ],
    y_axis_title=(
        "Output relative to<br>"
        "pre-Haiyan baseline (%)"
    ),
)

fig_roman_normalized.update_yaxes(type='log',row=2, col=1)
fig_roman_normalized.show()

In [ ]:
# ============================================================
# 8D. RELIABILITY-QUALIFIED PROCESSING SEQUENCE
# ============================================================

# ------------------------------------------------------------
# RQ 1:
# Daily direct, reliability-qualified and gap-filled NTL
# ------------------------------------------------------------

fig_rq_raw = plot_sc_timeseries(
    title=(""
        # "RQ diagnostic 1: daily direct, qualified "
        # "and gap-filled NTL"
    ),
    ntl_panel_title=(""
        # "Daily regional mean radiance on native method support"
    ),
    sc_series=[
        {
            "label": "RQ NTL SC",
            "x": rq_daily.index,
            "y": rq_daily["sc_pct"],
        },
    ],
    line_series=[
        {
            "name": "DNB-BRDF",
            "x": roman_daily.index,
            "y": roman_daily["mean_ntl"],
            "color": DNB_COLOR,
            "width": 1.5,
            "mode": "lines",
            "opacity": 0.75,
            "unit": "nW cm⁻² sr⁻¹",
        },
        {
            "name": "Reliability-qualified NTL",
            "x": rq_daily.index,
            "y": rq_daily["mean_ntl"],
            "color": RQ_COLOR,
            "width": 2.0,
            "mode": "lines+markers",
            "marker_size": 3,
            "unit": "nW cm⁻² sr⁻¹",
        },
        {
            "name": "Gap-filled",
            "x": gap_filled_daily.index,
            "y": gap_filled_daily["mean_ntl"],
            "color": GAP_FILLED_COLOR,
            "width": 1.8,
            "mode": "lines",
            "opacity": 0.75,
            "unit": "nW cm⁻² sr⁻¹",
        },
    ],
    y_axis_title=(
        "Mean DNB-BRDF<br>"
        "(nW cm⁻² sr⁻¹)"
    ),
)

fig_rq_raw.update_yaxes(type='log',row=2, col=1)
fig_rq_raw.show()


In [ ]:
# ------------------------------------------------------------
# RQ 2:
# Raw daily RQ DNB-BRDF versus four-day aggregation
# ------------------------------------------------------------

fig_rq_aggregation = plot_sc_timeseries(
    title="",
    ntl_panel_title=(""
        # "Raw daily RQ DNB-BRDF versus four-day mean"
    ),
    sc_series=[
        {
            "label": "RQ NTL SC",
            "x": rq_profile["date_start"],
            "y": rq_profile["spatial_coverage_pct"],
        },
    ],
    line_series=[
        {
            "name": "Reliability-qualified NTL",
            "x": rq_daily.index,
            "y": rq_daily["mean_ntl"],
            "color": RQ_COLOR,
            "width": 1.4,
            "mode": "lines+markers",
            "marker_size": 3,
            "opacity": 0.65,
            "unit": "nW cm⁻² sr⁻¹",
        },
        {
            "name": "Reliability-qualified NTL (4D)",
            "x": rq_profile["date_start"],
            "y": rq_profile["raw_mean_ntl"],
            "color": RQ_FOUR_DAY_COLOR,
            "width": 2.8,
            "mode": "lines+markers",
            "marker_size": 5,
            "shape": "hv",
            "unit": "nW cm⁻² sr⁻¹",
        },
    ],
    y_axis_title=(
        "Mean DNB-BRDF<br>"
        "(nW cm⁻² sr⁻¹)"
    ),
)

fig_rq_aggregation.show()

In [ ]:
# ------------------------------------------------------------
# RQ 3:
# Normalized DNB, RQ DNB and NGCP with RQ SC
# ------------------------------------------------------------

fig_rq_normalized = plot_sc_timeseries(
    title=(""
        # "RQ diagnostic 3: baseline-normalized comparison"
    ),
    ntl_panel_title=(""
        # "Four-day direct DNB, RQ DNB and NGCP"
    ),
    sc_series=[
        {
            "label": "RQ NTL SC",
            "x": rq_profile["date_start"],
            "y": rq_profile[
                "spatial_coverage_pct"
            ],
        },
    ],
    line_series=[
        {
            "name": "DNB-BRDF (4D)",
            "x": roman_profile["date_start"],
            "y": roman_profile["recovery_pct"],
            "color": DNB_FOUR_DAY_COLOR,
            "width": 2.5,
            "mode": "lines+markers",
            "marker_size": 5,
            "shape": "hv",
            "unit": "%",
        },
        {
            "name": "Reliability-qualified NTL (4D)",
            "x": rq_profile["date_start"],
            "y": rq_profile["recovery_pct"],
            "color": RQ_FOUR_DAY_COLOR,
            "width": 2.8,
            "mode": "lines+markers",
            "marker_size": 5,
            "shape": "hv",
            "unit": "%",
        },
        {
            "name": "NGCP 1 AM (4D)",
            "x": ngcp_profile["date_start"],
            "y": ngcp_profile["recovery_pct"],
            "color": NGCP_COLOR,
            "width": 2.5,
            "mode": "lines+markers",
            "marker_size": 4,
            "shape": "hv",
            "unit": "%",
        },
    ],
    y_axis_title=(
        "Output relative to<br>"
        "pre-Haiyan baseline (%)"
    ),
)

fig_rq_normalized.update_yaxes(type='log',row=2, col=1)
fig_rq_normalized.show()


In [ ]:
# ============================================================
# 8E. FINAL COMPARISON WITH BOTH SC SERIES
# ============================================================

fig_final_comparison = plot_sc_timeseries(
    title=(""
        # "Final four-day NTL and NGCP comparison"
    ),
    ntl_panel_title=(""
        # "Baseline-normalized direct DNB, RQ DNB and NGCP"
    ),
    sc_series=[
        {
            "label": "DNB-BRDF SC",
            "x": roman_profile["date_start"],
            "y": roman_profile[
                "spatial_coverage_pct"
            ],
        },
        {
            "label": "RQ NTL SC",
            "x": rq_profile["date_start"],
            "y": rq_profile[
                "spatial_coverage_pct"
            ],
        },
    ],
    line_series=[
        {
            "name": "DNB-BRDF (4D)",
            "x": roman_profile["date_start"],
            "y": roman_profile["recovery_pct"],
            "color": DNB_FOUR_DAY_COLOR,
            "width": 2.5,
            "mode": "lines+markers",
            "marker_size": 5,
            "shape": "hv",
            "unit": "%",
        },
        {
            "name": "Reliability-qualified NTL (4D)",
            "x": rq_profile["date_start"],
            "y": rq_profile["recovery_pct"],
            "color": RQ_FOUR_DAY_COLOR,
            "width": 2.8,
            "mode": "lines+markers",
            "marker_size": 5,
            "shape": "hv",
            "unit": "%",
        },
        {
            "name": "NGCP 1 AM (4D)",
            "x": ngcp_profile["date_start"],
            "y": ngcp_profile["recovery_pct"],
            "color": NGCP_COLOR,
            "width": 2.5,
            "mode": "lines+markers",
            "marker_size": 4,
            "shape": "hv",
            "unit": "%",
        },
    ],
    y_axis_title=(
        "Output relative to<br>"
        "pre-Haiyan baseline (%)"
    ),
)

fig_final_comparison.update_yaxes(type='log',row=2, col=1)
fig_final_comparison.show()

In [ ]:
# ============================================================
# 8F. COMPREHENSIVE SUMMARY OF RECOVERY RESULTS
# ============================================================

# Half-open intervals prevent overlap between phases.
summary_periods = [
    (
        "Baseline",
        pd.Timestamp(BASELINE_START),
        pd.Timestamp(EVENT_DATE),
    ),
    (
        "Stage 1 (0–59 days)",
        pd.Timestamp(EVENT_DATE),
        pd.Timestamp(EVENT_DATE)
        + pd.Timedelta(days=60),
    ),
    (
        "Stage 2 (60–119 days)",
        pd.Timestamp(EVENT_DATE)
        + pd.Timedelta(days=60),
        pd.Timestamp(EVENT_DATE)
        + pd.Timedelta(days=120),
    ),
    (
        "Stage 3 (120–179 days)",
        pd.Timestamp(EVENT_DATE)
        + pd.Timedelta(days=120),
        pd.Timestamp(EVENT_DATE)
        + pd.Timedelta(days=180),
    ),
    (
        "Post-event (0–179 days)",
        pd.Timestamp(EVENT_DATE),
        pd.Timestamp(EVENT_DATE)
        + pd.Timedelta(days=180),
    ),
    (
        "Full analysis",
        pd.Timestamp(BASELINE_START),
        pd.Timestamp(EVENT_DATE)
        + pd.Timedelta(days=180),
    ),
]


# ------------------------------------------------------------
# Prepare the NGCP comparator
# ------------------------------------------------------------

ngcp_summary = (
    ngcp_profile[
        [
            "date_start",
            "recovery_pct",
        ]
    ]
    .rename(
        columns={
            "recovery_pct": "ngcp_recovery_pct",
        }
    )
    .copy()
)

ngcp_summary["date_start"] = pd.to_datetime(
    ngcp_summary["date_start"]
).dt.normalize()

ngcp_summary = (
    ngcp_summary
    .drop_duplicates(
        subset="date_start",
    )
    .sort_values("date_start")
)


# ------------------------------------------------------------
# Safe descriptive and agreement metrics
# ------------------------------------------------------------

def safe_correlation(
    x,
    y,
    method="pearson",
):
    paired_values = pd.DataFrame(
        {
            "x": pd.to_numeric(
                x,
                errors="coerce",
            ),
            "y": pd.to_numeric(
                y,
                errors="coerce",
            ),
        }
    ).replace(
        [np.inf, -np.inf],
        np.nan,
    ).dropna()

    if (
        len(paired_values) < 2
        or paired_values["x"].nunique() < 2
        or paired_values["y"].nunique() < 2
    ):
        return np.nan

    return paired_values["x"].corr(
        paired_values["y"],
        method=method,
    )


def safe_mean(values):
    values = pd.to_numeric(
        values,
        errors="coerce",
    )

    return (
        float(values.mean())
        if values.notna().any()
        else np.nan
    )


def safe_median(values):
    values = pd.to_numeric(
        values,
        errors="coerce",
    )

    return (
        float(values.median())
        if values.notna().any()
        else np.nan
    )


# ------------------------------------------------------------
# Calculate results by method and phase
# ------------------------------------------------------------

summary_rows = []

for method_name, method_profile in [
    (
        "Román-style direct DNB-BRDF",
        roman_profile,
    ),
    (
        "Reliability-qualified DNB-BRDF",
        rq_profile,
    ),
]:
    method_summary = (
        method_profile[
            [
                "date_start",
                "recovery_pct",
                "spatial_coverage_pct",
            ]
        ]
        .rename(
            columns={
                "recovery_pct": "ntl_recovery_pct",
                "spatial_coverage_pct": "sc_pct",
            }
        )
        .copy()
    )

    method_summary["date_start"] = pd.to_datetime(
        method_summary["date_start"]
    ).dt.normalize()

    method_summary = (
        method_summary
        .drop_duplicates(
            subset="date_start",
        )
        .sort_values("date_start")
    )

    for (
        period_name,
        period_start,
        period_end,
    ) in summary_periods:

        method_period = method_summary.loc[
            (
                method_summary["date_start"]
                >= period_start
            )
            & (
                method_summary["date_start"]
                < period_end
            )
        ].copy()

        ngcp_period = ngcp_summary.loc[
            (
                ngcp_summary["date_start"]
                >= period_start
            )
            & (
                ngcp_summary["date_start"]
                < period_end
            )
        ].copy()

        paired_period = (
            method_period
            .merge(
                ngcp_period,
                on="date_start",
                how="inner",
            )
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .dropna(
                subset=[
                    "ntl_recovery_pct",
                    "ngcp_recovery_pct",
                ]
            )
        )

        ntl_values = pd.to_numeric(
            method_period["ntl_recovery_pct"],
            errors="coerce",
        ).dropna()

        sc_values = pd.to_numeric(
            method_period["sc_pct"],
            errors="coerce",
        ).dropna()

        ngcp_values = pd.to_numeric(
            ngcp_period["ngcp_recovery_pct"],
            errors="coerce",
        ).dropna()

        differences = (
            paired_period["ntl_recovery_pct"]
            - paired_period["ngcp_recovery_pct"]
        )

        expected_composites = int(
            np.ceil(
                (
                    period_end
                    - period_start
                ).days
                / ROMAN_BLOCK_DAYS
            )
        )

        retained_composites = int(
            ntl_values.shape[0]
        )

        retained_pct = (
            100.0
            * retained_composites
            / expected_composites
            if expected_composites > 0
            else np.nan
        )

        summary_rows.append(
            {
                "Method": method_name,
                "Period": period_name,
                "Expected n": expected_composites,
                "NTL n": retained_composites,
                "NGCP n": int(
                    ngcp_values.shape[0]
                ),
                "Paired n": int(
                    paired_period.shape[0]
                ),
                "Retained (%)": retained_pct,
                "Mean NTL recovery (%)": (
                    safe_mean(ntl_values)
                ),
                "Median NTL recovery (%)": (
                    safe_median(ntl_values)
                ),
                "NTL recovery SD (%)": (
                    float(ntl_values.std(ddof=1))
                    if len(ntl_values) > 1
                    else np.nan
                ),
                "Minimum NTL recovery (%)": (
                    float(ntl_values.min())
                    if len(ntl_values) > 0
                    else np.nan
                ),
                "Maximum NTL recovery (%)": (
                    float(ntl_values.max())
                    if len(ntl_values) > 0
                    else np.nan
                ),
                "Mean NGCP recovery (%)": (
                    safe_mean(ngcp_values)
                ),
                "Median NGCP recovery (%)": (
                    safe_median(ngcp_values)
                ),
                "Pearson r": safe_correlation(
                    paired_period[
                        "ntl_recovery_pct"
                    ],
                    paired_period[
                        "ngcp_recovery_pct"
                    ],
                    method="pearson",
                ),
                "Spearman ρ": safe_correlation(
                    paired_period[
                        "ntl_recovery_pct"
                    ],
                    paired_period[
                        "ngcp_recovery_pct"
                    ],
                    method="spearman",
                ),
                "MAE vs NGCP (%)": (
                    float(
                        np.abs(differences).mean()
                    )
                    if len(differences) > 0
                    else np.nan
                ),
                "RMSE vs NGCP (%)": (
                    float(
                        np.sqrt(
                            np.mean(
                                differences ** 2
                            )
                        )
                    )
                    if len(differences) > 0
                    else np.nan
                ),
                "Mean bias vs NGCP (%)": (
                    safe_mean(differences)
                ),
                "Mean SC (%)": (
                    safe_mean(sc_values)
                ),
                "Median SC (%)": (
                    safe_median(sc_values)
                ),
                "Minimum SC (%)": (
                    float(sc_values.min())
                    if len(sc_values) > 0
                    else np.nan
                ),
            }
        )


results_summary_table = pd.DataFrame(
    summary_rows
)

period_order = [
    period[0]
    for period in summary_periods
]

results_summary_table["Period"] = pd.Categorical(
    results_summary_table["Period"],
    categories=period_order,
    ordered=True,
)

results_summary_table = (
    results_summary_table
    .sort_values(
        [
            "Method",
            "Period",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Display the comprehensive results table
# ------------------------------------------------------------

results_summary_style = (
    results_summary_table
    .style
    .format(
        {
            "Expected n": "{:.0f}",
            "NTL n": "{:.0f}",
            "NGCP n": "{:.0f}",
            "Paired n": "{:.0f}",
            "Retained (%)": "{:.1f}",
            "Mean NTL recovery (%)": "{:.1f}",
            "Median NTL recovery (%)": "{:.1f}",
            "NTL recovery SD (%)": "{:.1f}",
            "Minimum NTL recovery (%)": "{:.1f}",
            "Maximum NTL recovery (%)": "{:.1f}",
            "Mean NGCP recovery (%)": "{:.1f}",
            "Median NGCP recovery (%)": "{:.1f}",
            "Pearson r": "{:.3f}",
            "Spearman ρ": "{:.3f}",
            "MAE vs NGCP (%)": "{:.1f}",
            "RMSE vs NGCP (%)": "{:.1f}",
            "Mean bias vs NGCP (%)": "{:+.1f}",
            "Mean SC (%)": "{:.1f}",
            "Median SC (%)": "{:.1f}",
            "Minimum SC (%)": "{:.1f}",
        },
        na_rep="—",
    )
    .set_caption(
        "Román-style and reliability-qualified recovery "
        "summary by analysis period"
    )
    .set_properties(
        **{
            "text-align": "center",
            "white-space": "nowrap",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("font-size", "16px"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                    ("margin-bottom", "8px"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#243B5A"),
                    ("color", "white"),
                    ("text-align", "center"),
                ],
            },
        ]
    )
)

display(results_summary_style)

In [ ]:
# ------------------------------------------------------------
# Display focused summary tables
# ------------------------------------------------------------

core_periods = [
    "Baseline",
    "Stage 1 (0–59 days)",
    "Stage 2 (60–119 days)",
    "Stage 3 (120–179 days)",
]

comparison_periods = core_periods + [
    "Post-event (0–179 days)",
    "Full analysis",
]


def display_results_table(
    table,
    caption,
    formats,
):
    styled_table = (
        table
        .style
        .format(
            formats,
            na_rep="—",
        )
        .hide(axis="index")
        .set_caption(caption)
        .set_properties(
            **{
                "text-align": "center",
                "white-space": "nowrap",
            }
        )
        .set_table_styles(
            [
                {
                    "selector": "caption",
                    "props": [
                        ("font-size", "16px"),
                        ("font-weight", "bold"),
                        ("text-align", "left"),
                        ("margin-bottom", "8px"),
                    ],
                },
                {
                    "selector": "th",
                    "props": [
                        ("background-color", "#243B5A"),
                        ("color", "white"),
                        ("text-align", "center"),
                    ],
                },
            ]
        )
    )

    display(styled_table)


# ============================================================
# TABLE 1. OBSERVABILITY AND DATA RETENTION
# ============================================================

observability_table = (
    results_summary_table.loc[
        results_summary_table["Period"].isin(
            comparison_periods
        ),
        [
            "Method",
            "Period",
            "Expected n",
            "NTL n",
            "Paired n",
            "Retained (%)",
            "Mean SC (%)",
            "Median SC (%)",
            "Minimum SC (%)",
        ],
    ]
    .rename(
        columns={
            "Expected n": "Expected composites",
            "NTL n": "Retained NTL composites",
            "Paired n": "Paired with NGCP",
        }
    )
    .reset_index(drop=True)
)

display_results_table(
    table=observability_table,
    caption=(
        "Table 1. Four-day composite availability and "
        "spatial completeness"
    ),
    formats={
        "Expected composites": "{:.0f}",
        "Retained NTL composites": "{:.0f}",
        "Paired with NGCP": "{:.0f}",
        "Retained (%)": "{:.1f}",
        "Mean SC (%)": "{:.1f}",
        "Median SC (%)": "{:.1f}",
        "Minimum SC (%)": "{:.1f}",
    },
)


In [ ]:


# ============================================================
# TABLE 2. MEDIAN RECOVERY BY PHASE
# ============================================================

phase_ntl_recovery = (
    results_summary_table.loc[
        results_summary_table["Period"].isin(
            core_periods
        ),
        [
            "Method",
            "Period",
            "Median NTL recovery (%)",
        ],
    ]
    .pivot(
        index="Method",
        columns="Period",
        values="Median NTL recovery (%)",
    )
    .reindex(columns=core_periods)
)

phase_ngcp_recovery = (
    results_summary_table.loc[
        results_summary_table["Period"].isin(
            core_periods
        )
    ]
    .groupby(
        "Period",
        observed=True,
    )["Median NGCP recovery (%)"]
    .first()
    .reindex(core_periods)
    .to_frame()
    .T
)

phase_ngcp_recovery.index = [
    "NGCP 1 AM demand"
]

phase_recovery_table = (
    pd.concat(
        [
            phase_ntl_recovery,
            phase_ngcp_recovery,
        ],
        axis=0,
    )
    .rename_axis("Series")
    .reset_index()
)

display_results_table(
    table=phase_recovery_table,
    caption=(
        "Table 2. Median output relative to the "
        "pre-Haiyan baseline by phase (%)"
    ),
    formats={
        period: "{:.1f}"
        for period in core_periods
    },
)


In [ ]:
# ============================================================
# TABLE 3. WITHIN-PHASE NTL VARIABILITY
# ============================================================

recovery_variability_table = (
    results_summary_table.loc[
        results_summary_table["Period"].isin(
            core_periods
        ),
        [
            "Method",
            "Period",
            "NTL n",
            "Mean NTL recovery (%)",
            "Median NTL recovery (%)",
            "NTL recovery SD (%)",
            "Minimum NTL recovery (%)",
            "Maximum NTL recovery (%)",
        ],
    ]
    .rename(
        columns={
            "NTL n": "n",
            "Mean NTL recovery (%)": "Mean (%)",
            "Median NTL recovery (%)": "Median (%)",
            "NTL recovery SD (%)": "SD (%)",
            "Minimum NTL recovery (%)": "Minimum (%)",
            "Maximum NTL recovery (%)": "Maximum (%)",
        }
    )
    .reset_index(drop=True)
)

display_results_table(
    table=recovery_variability_table,
    caption=(
        "Table 3. Distribution and variability of "
        "NTL-derived recovery by phase"
    ),
    formats={
        "n": "{:.0f}",
        "Mean (%)": "{:.1f}",
        "Median (%)": "{:.1f}",
        "SD (%)": "{:.1f}",
        "Minimum (%)": "{:.1f}",
        "Maximum (%)": "{:.1f}",
    },
)


In [ ]:
# ============================================================
# TABLE 4. AGREEMENT WITH NGCP
# ============================================================

agreement_table = (
    results_summary_table.loc[
        results_summary_table["Period"].isin(
            comparison_periods
        ),
        [
            "Method",
            "Period",
            "Paired n",
            "Pearson r",
            "Spearman ρ",
            "MAE vs NGCP (%)",
            "RMSE vs NGCP (%)",
            "Mean bias vs NGCP (%)",
        ],
    ]
    .rename(
        columns={
            "Paired n": "n",
            "MAE vs NGCP (%)": "MAE (%)",
            "RMSE vs NGCP (%)": "RMSE (%)",
            "Mean bias vs NGCP (%)": "Mean bias (%)",
        }
    )
    .reset_index(drop=True)
)

display_results_table(
    table=agreement_table,
    caption=(
        "Table 4. Agreement between four-day NTL "
        "recovery and NGCP demand"
    ),
    formats={
        "n": "{:.0f}",
        "Pearson r": "{:.3f}",
        "Spearman ρ": "{:.3f}",
        "MAE (%)": "{:.1f}",
        "RMSE (%)": "{:.1f}",
        "Mean bias (%)": "{:+.1f}",
    },
)

In [ ]:
# ============================================================
# TABLES 3–4. SIDE-BY-SIDE METHOD COMPARISONS
# ============================================================

period_labels = {
    "Baseline": "Baseline",
    "Stage 1 (0–59 days)": "Stage 1 (0–59 d)",
    "Stage 2 (60–119 days)": "Stage 2 (60–119 d)",
    "Stage 3 (120–179 days)": "Stage 3 (120–179 d)",
    "Post-event (0–179 days)": "Post-event (0–179 d)",
    "Full analysis": "Full analysis",
}

method_labels = {
    "Román-style direct DNB-BRDF": "Román-style",
    "Reliability-qualified DNB-BRDF": "Reliability-qualified",
}


def build_wide_comparison_table(
    periods,
    metric_columns,
    metric_labels,
):
    table_source = results_summary_table.loc[
        results_summary_table["Period"].isin(
            periods
        ),
        [
            "Method",
            "Period",
            *metric_columns,
        ],
    ].copy()

    table_source["Method"] = table_source[
        "Method"
    ].replace(method_labels)

    table_source["Period"] = (
        table_source["Period"]
        .astype(str)
        .replace(period_labels)
    )

    wide_table = table_source.pivot(
        index="Method",
        columns="Period",
        values=metric_columns,
    )

    # Pivot creates Metric → Period.
    # Reverse this to Period → Metric.
    wide_table = wide_table.swaplevel(
        0,
        1,
        axis=1,
    )

    wide_table.columns = pd.MultiIndex.from_tuples(
        [
            (
                period,
                metric_labels[metric],
            )
            for period, metric in wide_table.columns
        ],
        names=[
            "Period",
            "Metric",
        ],
    )

    ordered_columns = pd.MultiIndex.from_product(
        [
            [
                period_labels[period]
                for period in periods
            ],
            [
                metric_labels[metric]
                for metric in metric_columns
            ],
        ],
        names=[
            "Period",
            "Metric",
        ],
    )

    wide_table = wide_table.reindex(
        columns=ordered_columns
    )

    wide_table = wide_table.reindex(
        [
            "Román-style",
            "Reliability-qualified",
        ]
    )

    wide_table.index.name = "Method"

    return wide_table


def display_wide_comparison_table(
    table,
    caption,
    metric_formats,
):
    column_formats = {
        column: metric_formats[column[1]]
        for column in table.columns
    }

    def shade_method_row(row):
        if row.name == "Román-style":
            background = "#FFF3E6"
        else:
            background = "#E8F5EF"

        return [
            f"background-color: {background}"
            for _ in row
        ]

    styled_table = (
        table
        .style
        .format(
            column_formats,
            na_rep="—",
        )
        .apply(
            shade_method_row,
            axis=1,
        )
        .set_caption(caption)
        .set_properties(
            **{
                "text-align": "center",
                "white-space": "nowrap",
                "border": "1px solid #D5DCE5",
            }
        )
        .set_table_styles(
            [
                {
                    "selector": "caption",
                    "props": [
                        ("font-size", "16px"),
                        ("font-weight", "bold"),
                        ("text-align", "left"),
                        ("margin-bottom", "8px"),
                    ],
                },
                {
                    "selector": "th.col_heading.level0",
                    "props": [
                        ("background-color", "#243B5A"),
                        ("color", "white"),
                        ("font-weight", "bold"),
                        ("text-align", "center"),
                        ("border", "1px solid white"),
                    ],
                },
                {
                    "selector": "th.col_heading.level1",
                    "props": [
                        ("background-color", "#DCE5F0"),
                        ("color", "#243B5A"),
                        ("font-weight", "bold"),
                        ("text-align", "center"),
                        ("border", "1px solid white"),
                    ],
                },
                {
                    "selector": "th.row_heading",
                    "props": [
                        ("background-color", "#F3F5F8"),
                        ("color", "#243B5A"),
                        ("font-weight", "bold"),
                        ("text-align", "left"),
                        ("padding", "6px 10px"),
                    ],
                },
                {
                    "selector": "th.index_name",
                    "props": [
                        ("background-color", "#243B5A"),
                        ("color", "white"),
                        ("font-weight", "bold"),
                        ("text-align", "left"),
                    ],
                },
            ]
        )
    )

    display(styled_table)


# ============================================================
# TABLE 3A. PHASE RECOVERY LEVEL
# ============================================================

table_3a_metrics = [
    "NTL n",
    "Mean NTL recovery (%)",
    "Median NTL recovery (%)",
]

table_3a_labels = {
    "NTL n": "n",
    "Mean NTL recovery (%)": "Mean (%)",
    "Median NTL recovery (%)": "Median (%)",
}

table_3a = build_wide_comparison_table(
    periods=core_periods,
    metric_columns=table_3a_metrics,
    metric_labels=table_3a_labels,
)

display_wide_comparison_table(
    table=table_3a,
    caption=(
        "Table 3a. NTL-derived recovery level by phase"
    ),
    metric_formats={
        "n": "{:.0f}",
        "Mean (%)": "{:.1f}",
        "Median (%)": "{:.1f}",
    },
)


# ============================================================
# TABLE 3B. WITHIN-PHASE VARIABILITY
# ============================================================

table_3b_metrics = [
    "NTL recovery SD (%)",
    "Minimum NTL recovery (%)",
    "Maximum NTL recovery (%)",
]

table_3b_labels = {
    "NTL recovery SD (%)": "SD (%)",
    "Minimum NTL recovery (%)": "Minimum (%)",
    "Maximum NTL recovery (%)": "Maximum (%)",
}

table_3b = build_wide_comparison_table(
    periods=core_periods,
    metric_columns=table_3b_metrics,
    metric_labels=table_3b_labels,
)

display_wide_comparison_table(
    table=table_3b,
    caption=(
        "Table 3b. Within-phase variability of "
        "NTL-derived recovery"
    ),
    metric_formats={
        "SD (%)": "{:.1f}",
        "Minimum (%)": "{:.1f}",
        "Maximum (%)": "{:.1f}",
    },
)


# ============================================================
# TABLE 4A. PHASE-SPECIFIC ASSOCIATION WITH NGCP
# ============================================================

table_4a_metrics = [
    "Paired n",
    "Pearson r",
    "Spearman ρ",
]

table_4a_labels = {
    "Paired n": "n",
    "Pearson r": "Pearson r",
    "Spearman ρ": "Spearman ρ",
}

table_4a = build_wide_comparison_table(
    periods=core_periods,
    metric_columns=table_4a_metrics,
    metric_labels=table_4a_labels,
)

display_wide_comparison_table(
    table=table_4a,
    caption=(
        "Table 4a. Phase-specific association between "
        "NTL recovery and NGCP demand"
    ),
    metric_formats={
        "n": "{:.0f}",
        "Pearson r": "{:.3f}",
        "Spearman ρ": "{:.3f}",
    },
)


# ============================================================
# TABLE 4B. PHASE-SPECIFIC ERROR RELATIVE TO NGCP
# ============================================================

table_4b_metrics = [
    "MAE vs NGCP (%)",
    "RMSE vs NGCP (%)",
    "Mean bias vs NGCP (%)",
]

table_4b_labels = {
    "MAE vs NGCP (%)": "MAE (%)",
    "RMSE vs NGCP (%)": "RMSE (%)",
    "Mean bias vs NGCP (%)": "Mean bias (%)",
}

table_4b = build_wide_comparison_table(
    periods=core_periods,
    metric_columns=table_4b_metrics,
    metric_labels=table_4b_labels,
)

display_wide_comparison_table(
    table=table_4b,
    caption=(
        "Table 4b. Phase-specific NTL error relative "
        "to NGCP demand"
    ),
    metric_formats={
        "MAE (%)": "{:.1f}",
        "RMSE (%)": "{:.1f}",
        "Mean bias (%)": "{:+.1f}",
    },
)


# ============================================================
# TABLE 4C. OVERALL POST-EVENT AND FULL-PERIOD AGREEMENT
# ============================================================

overall_periods = [
    "Post-event (0–179 days)",
    "Full analysis",
]

table_4c_metrics = [
    "Paired n",
    "Pearson r",
    "Spearman ρ",
    "MAE vs NGCP (%)",
    "RMSE vs NGCP (%)",
    "Mean bias vs NGCP (%)",
]

table_4c_labels = {
    "Paired n": "n",
    "Pearson r": "Pearson r",
    "Spearman ρ": "Spearman ρ",
    "MAE vs NGCP (%)": "MAE (%)",
    "RMSE vs NGCP (%)": "RMSE (%)",
    "Mean bias vs NGCP (%)": "Mean bias (%)",
}

table_4c = build_wide_comparison_table(
    periods=overall_periods,
    metric_columns=table_4c_metrics,
    metric_labels=table_4c_labels,
)

display_wide_comparison_table(
    table=table_4c,
    caption=(
        "Table 4c. Overall agreement between NTL "
        "recovery and NGCP demand"
    ),
    metric_formats={
        "n": "{:.0f}",
        "Pearson r": "{:.3f}",
        "Spearman ρ": "{:.3f}",
        "MAE (%)": "{:.1f}",
        "RMSE (%)": "{:.1f}",
        "Mean bias (%)": "{:+.1f}",
    },
)

In [ ]:
# ============================================================
# 10. TACLOBAN BASELINE AND STAGE NTL MAPS
# ============================================================

TACLOBAN_X_RANGE = (124.94, 125.10)
TACLOBAN_Y_RANGE = (11.14, 11.32)


def subset_tacloban(data_array):
    """Subset an xarray raster regardless of coordinate direction."""

    x_values = data_array["x"].values
    y_values = data_array["y"].values

    x_slice = (
        slice(*TACLOBAN_X_RANGE)
        if x_values[0] < x_values[-1]
        else slice(*TACLOBAN_X_RANGE[::-1])
    )

    y_slice = (
        slice(*TACLOBAN_Y_RANGE)
        if y_values[0] < y_values[-1]
        else slice(*TACLOBAN_Y_RANGE[::-1])
    )

    return data_array.sel(
        x=x_slice,
        y=y_slice,
    )


def calculate_stage_map(
    composites,
    profile,
    stage_start,
    stage_end,
):
    """Median of admissible four-day composites within a stage."""

    eligible = profile.loc[
        (profile["date_start"] >= stage_start)
        & (profile["date_end"] <= stage_end)
        & profile["recovery_pct"].notna()
    ]

    stage_blocks = (
        eligible["block"]
        .astype(int)
        .to_numpy()
    )

    if len(stage_blocks) == 0:
        return xr.full_like(
            composites.isel(block=0),
            np.nan,
        ).compute()

    return (
        composites
        .sel(block=stage_blocks)
        .median(
            dim="block",
            skipna=True,
        )
        .compute()
    )


roman_maps = {
    "Baseline": roman_ntl0.compute(),
}

rq_maps = {
    "Baseline": rq_ntl0.compute(),
}

for stage_name, (
    stage_start,
    stage_end,
) in list(STAGE_WINDOWS.items())[1:]:

    roman_maps[stage_name] = calculate_stage_map(
        roman_composites,
        roman_profile,
        stage_start,
        stage_end,
    )

    rq_maps[stage_name] = calculate_stage_map(
        rq_composites,
        rq_profile,
        stage_start,
        stage_end,
    )


roman_tacloban_maps = {
    stage_name: subset_tacloban(stage_map)
    for stage_name, stage_map in roman_maps.items()
}

rq_tacloban_maps = {
    stage_name: subset_tacloban(stage_map)
    for stage_name, stage_map in rq_maps.items()
}

all_map_values = []

for map_collection in (
    roman_tacloban_maps,
    rq_tacloban_maps,
):
    for stage_map in map_collection.values():
        values = np.asarray(
            stage_map.values,
            dtype=float,
        )

        values = values[
            np.isfinite(values)
        ]

        if values.size > 0:
            all_map_values.append(values)

if not all_map_values:
    raise ValueError(
        "No valid Tacloban NTL values were available for mapping."
    )

all_map_values = np.concatenate(
    all_map_values
)

radiance_color_max = max(
    float(
        np.nanpercentile(
            all_map_values,
            98,
        )
    ),
    1.0,
)

stage_names = list(STAGE_WINDOWS)

subplot_titles = [
    f"Román | {stage_name}"
    for stage_name in stage_names
] + [
    f"RQ | {stage_name}"
    for stage_name in stage_names
]

fig_maps = make_subplots(
    rows=2,
    cols=4,
    horizontal_spacing=0.035,
    vertical_spacing=0.10,
    subplot_titles=subplot_titles,
)

for row_number, map_collection in (
    (1, roman_tacloban_maps),
    (2, rq_tacloban_maps),
):
    for column_number, stage_name in enumerate(
        stage_names,
        start=1,
    ):
        stage_map = map_collection[stage_name]

        fig_maps.add_trace(
            go.Heatmap(
                x=stage_map["x"].values,
                y=stage_map["y"].values,
                z=stage_map.values,
                coloraxis="coloraxis",
                zsmooth=False,
                hovertemplate=(
                    "Longitude: %{x:.4f}<br>"
                    "Latitude: %{y:.4f}<br>"
                    "NTL: %{z:.2f} "
                    "nW cm⁻² sr⁻¹"
                    "<extra></extra>"
                ),
            ),
            row=row_number,
            col=column_number,
        )

        axis_number = (
            (row_number - 1) * 4
            + column_number
        )

        x_axis_reference = (
            "x"
            if axis_number == 1
            else f"x{axis_number}"
        )

        fig_maps.update_yaxes(
            scaleanchor=x_axis_reference,
            scaleratio=1,
            row=row_number,
            col=column_number,
        )

        fig_maps.update_xaxes(
            range=list(TACLOBAN_X_RANGE),
            row=row_number,
            col=column_number,
        )

        fig_maps.update_yaxes(
            range=list(TACLOBAN_Y_RANGE),
            row=row_number,
            col=column_number,
        )

fig_maps.update_xaxes(
    title_text="Longitude",
    row=2,
)

fig_maps.update_yaxes(
    title_text="Latitude",
    col=1,
)

fig_maps.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="#D9D9D9",
    width=1500,
    height=760,
    title=dict(
        text=(
            "Tacloban baseline and stage-specific "
            "nighttime-light radiance"
        ),
        x=0.5,
        y=0.98,
        xanchor="center",
        font=dict(size=24),
    ),
    coloraxis=dict(
        colorscale="Inferno",
        cmin=0,
        cmax=radiance_color_max,
        colorbar=dict(
            title=dict(
                text="NTL<br>nW cm⁻² sr⁻¹"
            ),
            thickness=18,
            len=0.82,
        ),
    ),
    font=dict(
        family="Arial",
        size=13,
        color="#243B5A",
    ),
    margin=dict(
        l=70,
        r=110,
        t=120,
        b=70,
    ),
)

fig_maps.show()

In [ ]:
# ============================================================
# 10. TACLOBAN BASELINE AND STAGE NTL MAPS
#     Román: no GHSL filtering
#     RQ: GHSL G7 with non-G7 land shown in gray
# ============================================================

import geopandas as gpd


# ------------------------------------------------------------
# 10.1 Load the regional boundary
# ------------------------------------------------------------

REGION_SHAPEFILE = Path(
    "../datasets/boundaries/regions/Regions.shp"
)

if not REGION_SHAPEFILE.exists():
    raise FileNotFoundError(
        f"Regional boundary shapefile not found:\n"
        f"{REGION_SHAPEFILE}"
    )

region_boundary = (
    gpd.read_file(REGION_SHAPEFILE)
    .to_crs("EPSG:4326")
)


# ------------------------------------------------------------
# 10.2 Tacloban spatial extent
# ------------------------------------------------------------

TACLOBAN_X_RANGE = (124.94, 125.10)
TACLOBAN_Y_RANGE = (11.14, 11.32)


def subset_tacloban(data_array):
    """Subset a raster regardless of coordinate direction."""

    x_values = data_array["x"].values
    y_values = data_array["y"].values

    x_slice = (
        slice(*TACLOBAN_X_RANGE)
        if x_values[0] < x_values[-1]
        else slice(*TACLOBAN_X_RANGE[::-1])
    )

    y_slice = (
        slice(*TACLOBAN_Y_RANGE)
        if y_values[0] < y_values[-1]
        else slice(*TACLOBAN_Y_RANGE[::-1])
    )

    return data_array.sel(
        x=x_slice,
        y=y_slice,
    )


# ------------------------------------------------------------
# 10.3 Calculate stage-level median NTL
# ------------------------------------------------------------

def calculate_stage_map(
    composites,
    profile,
    stage_start,
    stage_end,
):
    """Median of admissible four-day composites within a stage."""

    eligible = profile.loc[
        (profile["date_start"] >= stage_start)
        & (profile["date_end"] <= stage_end)
        & profile["recovery_pct"].notna()
    ]

    stage_blocks = (
        eligible["block"]
        .astype(int)
        .to_numpy()
    )

    if len(stage_blocks) == 0:
        return xr.full_like(
            composites.isel(block=0),
            np.nan,
        ).compute()

    return (
        composites
        .sel(block=stage_blocks)
        .median(
            dim="block",
            skipna=True,
        )
        .compute()
    )


roman_maps = {
    "Baseline": roman_ntl0.compute(),
}

rq_maps = {
    "Baseline": rq_ntl0.compute(),
}

for stage_name, (
    stage_start,
    stage_end,
) in list(STAGE_WINDOWS.items())[1:]:

    roman_maps[stage_name] = calculate_stage_map(
        composites=roman_composites,
        profile=roman_profile,
        stage_start=stage_start,
        stage_end=stage_end,
    )

    rq_maps[stage_name] = calculate_stage_map(
        composites=rq_composites,
        profile=rq_profile,
        stage_start=stage_start,
        stage_end=stage_end,
    )


roman_tacloban_maps = {
    stage_name: subset_tacloban(stage_map)
    for stage_name, stage_map in roman_maps.items()
}

rq_tacloban_maps = {
    stage_name: subset_tacloban(stage_map)
    for stage_name, stage_map in rq_maps.items()
}


# ------------------------------------------------------------
# 10.4 Construct the land mask from the shapefile
# ------------------------------------------------------------

reference_map = roman_tacloban_maps["Baseline"]

x_coordinates = reference_map["x"].values
y_coordinates = reference_map["y"].values

longitude_grid, latitude_grid = np.meshgrid(
    x_coordinates,
    y_coordinates,
)

pixel_centres = gpd.GeoSeries(
    gpd.points_from_xy(
        longitude_grid.ravel(),
        latitude_grid.ravel(),
    ),
    crs="EPSG:4326",
)

land_geometry = region_boundary.geometry.unary_union

land_values = (
    pixel_centres
    .intersects(land_geometry)
    .to_numpy()
    .reshape(
        len(y_coordinates),
        len(x_coordinates),
    )
)

land_mask_tacloban = xr.DataArray(
    land_values,
    dims=("y", "x"),
    coords={
        "y": y_coordinates,
        "x": x_coordinates,
    },
)

g7_tacloban = (
    subset_tacloban(g7_mask)
    .fillna(False)
    .astype(bool)
)

# Román: retain every land pixel; no GHSL filtering.
roman_display_maps = {
    stage_name: stage_map.where(
        land_mask_tacloban
    )
    for stage_name, stage_map
    in roman_tacloban_maps.items()
}

# RQ only: retain GHSL G7 land pixels.
rq_display_mask = (
    land_mask_tacloban
    & g7_tacloban
)

rq_display_maps = {
    stage_name: stage_map.where(
        rq_display_mask
    )
    for stage_name, stage_map
    in rq_tacloban_maps.items()
}


# ------------------------------------------------------------
# 10.5 Extract the regional boundary lines
# ------------------------------------------------------------

tacloban_boundary = region_boundary.cx[
    TACLOBAN_X_RANGE[0]:TACLOBAN_X_RANGE[1],
    TACLOBAN_Y_RANGE[0]:TACLOBAN_Y_RANGE[1],
].copy()


def extract_boundary_lines(geodataframe):
    """Return polygon exterior coordinates for Plotly."""

    boundary_lines = []

    for geometry in geodataframe.geometry:
        if geometry is None or geometry.is_empty:
            continue

        if geometry.geom_type == "Polygon":
            polygons = [geometry]

        elif geometry.geom_type == "MultiPolygon":
            polygons = list(geometry.geoms)

        else:
            continue

        for polygon in polygons:
            longitude, latitude = (
                polygon.exterior.xy
            )

            boundary_lines.append(
                (
                    np.asarray(longitude),
                    np.asarray(latitude),
                )
            )

    return boundary_lines


boundary_lines = extract_boundary_lines(
    tacloban_boundary
)


# ------------------------------------------------------------
# 10.6 Universal shared radiance scale
# ------------------------------------------------------------

all_map_values = []

for map_collection in (
    roman_display_maps,
    rq_display_maps,
):
    for stage_map in map_collection.values():
        values = np.asarray(
            stage_map.values,
            dtype=float,
        )

        values = values[
            np.isfinite(values)
        ]

        if values.size > 0:
            all_map_values.append(values)

if not all_map_values:
    raise ValueError(
        "No valid Tacloban NTL values were available."
    )

all_map_values = np.concatenate(
    all_map_values
)

radiance_color_max = max(
    float(
        np.nanpercentile(
            all_map_values,
            98,
        )
    ),
    1.0,
)

print(
    "Universal radiance range: "
    f"0–{radiance_color_max:.2f} nW cm⁻² sr⁻¹"
)


# ------------------------------------------------------------
# 10.7 Plot baseline and stage maps
# ------------------------------------------------------------

stage_names = list(STAGE_WINDOWS.keys())

subplot_titles = [
    f"Román | {stage_name}"
    for stage_name in stage_names
] + [
    f"RQ | {stage_name}"
    for stage_name in stage_names
]

fig_maps = make_subplots(
    rows=2,
    cols=4,
    horizontal_spacing=0.035,
    vertical_spacing=0.10,
    subplot_titles=subplot_titles,
)

# Used only under the RQ row.
# Land is gray; water remains transparent/white.
land_background = np.where(
    land_mask_tacloban.values,
    1.0,
    np.nan,
)

for row_number, map_collection in (
    (1, roman_display_maps),
    (2, rq_display_maps),
):
    for column_number, stage_name in enumerate(
        stage_names,
        start=1,
    ):
        stage_map = map_collection[stage_name]

        # Gray non-G7 land background for RQ only.
        if row_number == 2:
            fig_maps.add_trace(
                go.Heatmap(
                    x=x_coordinates,
                    y=y_coordinates,
                    z=land_background,
                    colorscale=[
                        [0.0, "#C8C8C8"],
                        [1.0, "#C8C8C8"],
                    ],
                    zmin=0,
                    zmax=1,
                    showscale=False,
                    hoverinfo="skip",
                ),
                row=row_number,
                col=column_number,
            )

        # NTL radiance
        hover_text = np.where(
            np.isfinite(stage_map.values),
            np.round(stage_map.values, 2).astype(str),
            "",
        )

        fig_maps.add_trace(
            go.Heatmap(
                x=stage_map["x"].values,
                y=stage_map["y"].values,
                z=stage_map.values,
                text=hover_text,
                coloraxis="coloraxis",
                zsmooth=False,
                hovertemplate=(
                    "Longitude: %{x:.4f}<br>"
                    "Latitude: %{y:.4f}<br>"
                    "NTL: %{text} nW cm⁻² sr⁻¹"
                    "<extra></extra>"
                ),
            ),
            row=row_number,
            col=column_number,
        )

        # Shapefile boundary overlay
        for longitude, latitude in boundary_lines:
            fig_maps.add_trace(
                go.Scatter(
                    x=longitude,
                    y=latitude,
                    mode="lines",
                    line=dict(
                        color="#303030",
                        width=1.3,
                    ),
                    hoverinfo="skip",
                    showlegend=False,
                ),
                row=row_number,
                col=column_number,
            )

        axis_number = (
            (row_number - 1) * 4
            + column_number
        )

        x_axis_reference = (
            "x"
            if axis_number == 1
            else f"x{axis_number}"
        )

        fig_maps.update_xaxes(
            range=list(TACLOBAN_X_RANGE),
            showgrid=False,
            zeroline=False,
            row=row_number,
            col=column_number,
        )

        fig_maps.update_yaxes(
            range=list(TACLOBAN_Y_RANGE),
            scaleanchor=x_axis_reference,
            scaleratio=1,
            showgrid=False,
            zeroline=False,
            row=row_number,
            col=column_number,
        )

fig_maps.update_xaxes(
    title_text="Longitude",
    row=2,
)

fig_maps.update_yaxes(
    title_text="Latitude",
    col=1,
)

fig_maps.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1200,
    height=600,
    coloraxis=dict(
        colorscale="Inferno",
        cmin=0,
        cmax=radiance_color_max,
        colorbar=dict(
            title=dict(
                text="NTL<br>nW cm⁻² sr⁻¹"
            ),
            x=1.015,
            xanchor="left",
            y=0.5,
            len=0.82,
            thickness=20,
            outlinewidth=0.8,
            outlinecolor="#555555",
        ),
    ),
    font=dict(
        family="Arial",
        size=13,
        color="#243B5A",
    ),
    margin=dict(
        l=70,
        r=130,
        t=135,
        b=70,
    ),
)

fig_maps.show()

In [ ]:

import geopandas as gpd


# ------------------------------------------------------------
# 10.1 Load the regional boundary
# ------------------------------------------------------------

REGION_SHAPEFILE = Path(
    "../datasets/boundaries/regions/Regions.shp"
)

if not REGION_SHAPEFILE.exists():
    raise FileNotFoundError(
        f"Regional boundary shapefile not found:\n"
        f"{REGION_SHAPEFILE}"
    )

region_boundary = (
    gpd.read_file(REGION_SHAPEFILE)
    .to_crs("EPSG:4326")
)


# ------------------------------------------------------------
# 10.2 Tacloban spatial extent
# ------------------------------------------------------------

TACLOBAN_X_RANGE = (124.94, 125.10)
TACLOBAN_Y_RANGE = (11.14, 11.32)


def subset_tacloban(data_array):
    """Subset a raster regardless of coordinate direction."""

    x_values = data_array["x"].values
    y_values = data_array["y"].values

    x_slice = (
        slice(*TACLOBAN_X_RANGE)
        if x_values[0] < x_values[-1]
        else slice(*TACLOBAN_X_RANGE[::-1])
    )

    y_slice = (
        slice(*TACLOBAN_Y_RANGE)
        if y_values[0] < y_values[-1]
        else slice(*TACLOBAN_Y_RANGE[::-1])
    )

    return data_array.sel(
        x=x_slice,
        y=y_slice,
    )


# ------------------------------------------------------------
# 10.3 Calculate stage-level median NTL
# ------------------------------------------------------------

def calculate_stage_map(
    composites,
    profile,
    stage_start,
    stage_end,
):
    """Median of admissible four-day composites within a stage."""

    eligible = profile.loc[
        (profile["date_start"] >= stage_start)
        & (profile["date_end"] <= stage_end)
        & profile["recovery_pct"].notna()
    ]

    stage_blocks = (
        eligible["block"]
        .astype(int)
        .to_numpy()
    )

    if len(stage_blocks) == 0:
        return xr.full_like(
            composites.isel(block=0),
            np.nan,
        ).compute()

    return (
        composites
        .sel(block=stage_blocks)
        .median(
            dim="block",
            skipna=True,
        )
        .compute()
    )


roman_maps = {
    "Baseline": roman_ntl0.compute(),
}

rq_maps = {
    "Baseline": rq_ntl0.compute(),
}

for stage_name, (
    stage_start,
    stage_end,
) in list(STAGE_WINDOWS.items())[1:]:

    roman_maps[stage_name] = calculate_stage_map(
        composites=roman_composites,
        profile=roman_profile,
        stage_start=stage_start,
        stage_end=stage_end,
    )

    rq_maps[stage_name] = calculate_stage_map(
        composites=rq_composites,
        profile=rq_profile,
        stage_start=stage_start,
        stage_end=stage_end,
    )


roman_tacloban_maps = {
    stage_name: subset_tacloban(stage_map)
    for stage_name, stage_map in roman_maps.items()
}

rq_tacloban_maps = {
    stage_name: subset_tacloban(stage_map)
    for stage_name, stage_map in rq_maps.items()
}


# ------------------------------------------------------------
# 10.4 Construct the land mask from the shapefile
# ------------------------------------------------------------

reference_map = roman_tacloban_maps["Baseline"]

x_coordinates = reference_map["x"].values
y_coordinates = reference_map["y"].values

longitude_grid, latitude_grid = np.meshgrid(
    x_coordinates,
    y_coordinates,
)

pixel_centres = gpd.GeoSeries(
    gpd.points_from_xy(
        longitude_grid.ravel(),
        latitude_grid.ravel(),
    ),
    crs="EPSG:4326",
)

land_geometry = region_boundary.geometry.union_all()

land_values = (
    pixel_centres
    .intersects(land_geometry)
    .to_numpy()
    .reshape(
        len(y_coordinates),
        len(x_coordinates),
    )
)

land_mask_tacloban = xr.DataArray(
    land_values,
    dims=("y", "x"),
    coords={
        "y": y_coordinates,
        "x": x_coordinates,
    },
)

g7_tacloban = (
    subset_tacloban(g7_mask)
    .fillna(False)
    .astype(bool)
)

# Román: retain every land pixel; no GHSL filtering.
roman_display_maps = {
    stage_name: stage_map.where(
        land_mask_tacloban
    )
    for stage_name, stage_map
    in roman_tacloban_maps.items()
}

# RQ only: retain GHSL G7 land pixels.
rq_display_mask = (
    land_mask_tacloban
    & g7_tacloban
)

rq_display_maps = {
    stage_name: stage_map.where(
        rq_display_mask
    )
    for stage_name, stage_map
    in rq_tacloban_maps.items()
}


In [ ]:
# ============================================================
# 10.6 PUBLIC-FACING RQ MAPS
#     Display the four existing RQ maps only
# ============================================================

stage_names = list(rq_display_maps.keys())

public_labels = [
    "Before Haiyan",
    "0–2 months",
    "2–4 months",
    "4–6 months",
]

if len(stage_names) != 4:
    raise ValueError(
        f"Expected four RQ maps, but found {len(stage_names)}."
    )


# ------------------------------------------------------------
# Shared colour scale from the four RQ maps
# ------------------------------------------------------------

all_values = []

for stage_map in rq_display_maps.values():
    values = np.asarray(
        stage_map.values,
        dtype=float,
    )

    values = values[np.isfinite(values)]

    if values.size:
        all_values.append(values)

if not all_values:
    raise ValueError(
        "No valid nighttime-lights values were available."
    )

all_values = np.concatenate(all_values)

color_max = max(
    float(np.nanpercentile(all_values, 98)),
    1.0,
)


# ------------------------------------------------------------
# Four-panel figure
# ------------------------------------------------------------

fig_maps = make_subplots(
    rows=1,
    cols=4,
    horizontal_spacing=0.012,
    subplot_titles=[
        f"<b>{label}</b>"
        for label in public_labels
    ],
)

for column_number, stage_name in enumerate(
    stage_names,
    start=1,
):
    stage_map = rq_display_maps[stage_name]

    # Nighttime lights
    fig_maps.add_trace(
        go.Heatmap(
            x=stage_map["x"].values,
            y=stage_map["y"].values,
            z=stage_map.values,
            coloraxis="coloraxis",
            zsmooth=False,
            hoverinfo="skip",
        ),
        row=1,
        col=column_number,
    )

    # Boundary overlay
    for longitude, latitude in boundary_lines:
        fig_maps.add_trace(
            go.Scatter(
                x=longitude,
                y=latitude,
                mode="lines",
                line=dict(
                    color="#303030",
                    width=1.2,
                ),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column_number,
        )

    x_axis_reference = (
        "x"
        if column_number == 1
        else f"x{column_number}"
    )

    fig_maps.update_xaxes(
        range=list(TACLOBAN_X_RANGE),
        showticklabels=False,
        ticks="",
        title_text=None,
        showgrid=False,
        zeroline=False,
        showline=False,
        fixedrange=True,
        row=1,
        col=column_number,
    )

    fig_maps.update_yaxes(
        range=list(TACLOBAN_Y_RANGE),
        showticklabels=False,
        ticks="",
        title_text=None,
        showgrid=False,
        zeroline=False,
        showline=False,
        fixedrange=True,
        scaleanchor=x_axis_reference,
        scaleratio=1,
        constrain="domain",
        row=1,
        col=column_number,
    )


# ------------------------------------------------------------
# Public-facing formatting
# ------------------------------------------------------------

fig_maps.update_annotations(
    font=dict(
        family="Arial",
        size=17,
        color="#243B5A",
    )
)

fig_maps.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1200,
    height=315,
    coloraxis=dict(
        colorscale="Inferno",
        cmin=0,
        cmax=color_max,
        colorbar=dict(
            title=dict(
                text="<b>Nighttime<br>lights</b>",
                side="top",
                font=dict(size=14),
            ),
            tickmode="array",
            tickvals=[
                0,
                color_max,
            ],
            ticktext=[
                "Low",
                "High",
            ],
            tickfont=dict(size=13),
            x=1.01,
            xanchor="left",
            y=0.48,
            len=0.72,
            thickness=17,
            outlinewidth=0.8,
            outlinecolor="#555555",
        ),
    ),
    font=dict(
        family="Arial",
        size=13,
        color="#243B5A",
    ),
    margin=dict(
        l=5,
        r=95,
        t=55,
        b=5,
    ),
)

fig_maps.show()

In [ ]:
# ============================================================
# 10.6 PUBLIC-FACING RQ MAPS
#     Four reliability-qualified maps only
# ============================================================

stage_names = list(STAGE_WINDOWS.keys())

public_labels = [
    "Before Haiyan",
    "0–2 months",
    "2–4 months",
    "4–6 months",
]

if len(stage_names) != len(public_labels):
    raise ValueError(
        "Expected four stages: baseline and three recovery periods."
    )


# ------------------------------------------------------------
# Shared colour range using RQ maps only
# ------------------------------------------------------------

rq_map_values = []

for stage_map in rq_display_maps.values():
    values = np.asarray(
        stage_map.values,
        dtype=float,
    )

    values = values[np.isfinite(values)]

    if values.size > 0:
        rq_map_values.append(values)

if not rq_map_values:
    raise ValueError(
        "No valid Tacloban nighttime-lights values were available."
    )

rq_map_values = np.concatenate(rq_map_values)

radiance_color_max = max(
    float(np.nanpercentile(rq_map_values, 98)),
    1.0,
)


# ------------------------------------------------------------
# Four-panel public-facing figure
# ------------------------------------------------------------

fig_maps = make_subplots(
    rows=1,
    cols=4,
    horizontal_spacing=0.012,
    subplot_titles=public_labels,
)

# Light-grey land background outside the selected urban pixels.
land_background = np.where(
    land_mask_tacloban.values,
    1.0,
    np.nan,
)

for column_number, stage_name in enumerate(
    stage_names,
    start=1,
):
    stage_map = rq_display_maps[stage_name]

    # Land background
    fig_maps.add_trace(
        go.Heatmap(
            x=x_coordinates,
            y=y_coordinates,
            z=land_background,
            colorscale=[
                [0.0, "#E6E6E6"],
                [1.0, "#E6E6E6"],
            ],
            zmin=0,
            zmax=1,
            showscale=False,
            hoverinfo="skip",
        ),
        row=1,
        col=column_number,
    )

    # Nighttime lights
    fig_maps.add_trace(
        go.Heatmap(
            x=stage_map["x"].values,
            y=stage_map["y"].values,
            z=stage_map.values,
            coloraxis="coloraxis",
            zsmooth=False,
            hoverinfo="skip",
        ),
        row=1,
        col=column_number,
    )

    # Regional boundary
    for longitude, latitude in boundary_lines:
        fig_maps.add_trace(
            go.Scatter(
                x=longitude,
                y=latitude,
                mode="lines",
                line=dict(
                    color="#303030",
                    width=1.2,
                ),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column_number,
        )

    x_axis_reference = (
        "x"
        if column_number == 1
        else f"x{column_number}"
    )

    fig_maps.update_xaxes(
        range=list(TACLOBAN_X_RANGE),
        showticklabels=False,
        ticks="",
        title_text=None,
        showgrid=False,
        zeroline=False,
        showline=False,
        fixedrange=True,
        row=1,
        col=column_number,
    )

    fig_maps.update_yaxes(
        range=list(TACLOBAN_Y_RANGE),
        showticklabels=False,
        ticks="",
        title_text=None,
        showgrid=False,
        zeroline=False,
        showline=False,
        fixedrange=True,
        scaleanchor=x_axis_reference,
        scaleratio=1,
        row=1,
        col=column_number,
    )


fig_maps.update_annotations(
    font=dict(
        family="Arial",
        size=17,
        color="#243B5A",
    )
)

fig_maps.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1200,
    height=315,
    coloraxis=dict(
        colorscale="Inferno",
        cmin=0,
        cmax=radiance_color_max,
        colorbar=dict(
            title=dict(
                text="Nighttime<br>lights",
                side="top",
                font=dict(size=14),
            ),
            tickmode="array",
            tickvals=[
                0,
                radiance_color_max,
            ],
            ticktext=[
                "Low",
                "High",
            ],
            tickfont=dict(size=13),
            x=1.01,
            xanchor="left",
            y=0.48,
            len=0.72,
            thickness=17,
            outlinewidth=0.8,
            outlinecolor="#555555",
        ),
    ),
    font=dict(
        family="Arial",
        size=13,
        color="#243B5A",
    ),
    margin=dict(
        l=5,
        r=95,
        t=55,
        b=5,
    ),
)

fig_maps.show()

In [ ]:
# ============================================================
# 10.8 FOUR-DAY COMPOSITE PROGRESSION BY PHASE
# ============================================================

# Each row contains 15 consecutive four-day composites.
progression_rows = [
    (
        "Pre-event baseline\n(−60 to −1 d)",
        list(range(-15, 0)),
    ),
    (
        "Stage 1\n(0–59 d)",
        list(range(0, 15)),
    ),
    (
        "Stage 2\n(60–119 d)",
        list(range(15, 30)),
    ),
    (
        "Stage 3\n(120–179 d)",
        list(range(30, 45)),
    ),
]


# ------------------------------------------------------------
# Display masks
# ------------------------------------------------------------

roman_progression_mask = (
    land_mask_tacloban
    & subset_tacloban(
        roman_fixed_mask
    )
    .fillna(False)
    .astype(bool)
)

rq_progression_mask = (
    land_mask_tacloban
    & g7_tacloban
    & subset_tacloban(
        rq_fixed_mask
    )
    .fillna(False)
    .astype(bool)
)

progression_land_background = np.where(
    land_mask_tacloban.values,
    1.0,
    np.nan,
)


# ------------------------------------------------------------
# Prepare one composite map
# ------------------------------------------------------------

def prepare_progression_map(
    composite_cube,
    block_number,
    display_mask,
):
    """Extract and mask one Tacloban four-day composite."""

    available_blocks = np.asarray(
        composite_cube["block"].values,
        dtype=int,
    )

    if block_number not in available_blocks:
        return None

    composite_map = (
        subset_tacloban(
            composite_cube.sel(
                block=block_number
            )
        )
        .where(display_mask)
    )

    if hasattr(
        composite_map.data,
        "compute",
    ):
        composite_map = (
            composite_map.compute()
        )

    return composite_map


# ------------------------------------------------------------
# Prepare all Román and RQ snapshots
# ------------------------------------------------------------

roman_progression_maps = []
rq_progression_maps = []

for _, block_numbers in progression_rows:
    roman_progression_maps.append(
        [
            prepare_progression_map(
                composite_cube=roman_composites,
                block_number=block_number,
                display_mask=roman_progression_mask,
            )
            for block_number in block_numbers
        ]
    )

    rq_progression_maps.append(
        [
            prepare_progression_map(
                composite_cube=rq_composites,
                block_number=block_number,
                display_mask=rq_progression_mask,
            )
            for block_number in block_numbers
        ]
    )


# ------------------------------------------------------------
# One radiance scale shared by both figures
# ------------------------------------------------------------

progression_values = []

for map_grid in (
    roman_progression_maps,
    rq_progression_maps,
):
    for map_row in map_grid:
        for composite_map in map_row:
            if composite_map is None:
                continue

            values = np.asarray(
                composite_map.values,
                dtype=float,
            )

            values = values[
                np.isfinite(values)
            ]

            if values.size > 0:
                progression_values.append(
                    values
                )

if not progression_values:
    raise ValueError(
        "No valid four-day Tacloban maps "
        "were available."
    )

progression_values = np.concatenate(
    progression_values
)

progression_color_max = max(
    float(
        np.nanpercentile(
            progression_values,
            98,
        )
    ),
    1.0,
)

print(
    "Common progression radiance range: "
    f"0–{progression_color_max:.2f} "
    "nW cm⁻² sr⁻¹"
)


# ------------------------------------------------------------
# Combine boundary segments into one trace per panel
# ------------------------------------------------------------

progression_boundary_x = []
progression_boundary_y = []

for longitude, latitude in boundary_lines:
    progression_boundary_x.extend(
        [
            *longitude.tolist(),
            None,
        ]
    )

    progression_boundary_y.extend(
        [
            *latitude.tolist(),
            None,
        ]
    )


# ------------------------------------------------------------
# Plotting function
# ------------------------------------------------------------

def plot_composite_progression(
    map_grid,
    figure_title,
):
    """Plot 15 four-day snapshots for each analysis phase."""

    figure = make_subplots(
        rows=4,
        cols=15,
        shared_xaxes=True,
        shared_yaxes=True,
        horizontal_spacing=0.002,
        vertical_spacing=0.025,
        column_titles=[
            str(column_number)
            for column_number in range(
                1,
                16,
            )
        ],
    )

    longitude_centre = np.mean(
        TACLOBAN_X_RANGE
    )

    latitude_centre = np.mean(
        TACLOBAN_Y_RANGE
    )

    for row_number, (
        row_label,
        block_numbers,
    ) in enumerate(
        progression_rows,
        start=1,
    ):
        figure.update_yaxes(
            title_text=row_label,
            title_font=dict(
                size=12,
            ),
            title_standoff=4,
            row=row_number,
            col=1,
        )

        for column_number, block_number in enumerate(
            block_numbers,
            start=1,
        ):
            composite_map = map_grid[
                row_number - 1
            ][
                column_number - 1
            ]

            # Gray land background; water remains white.
            figure.add_trace(
                go.Heatmap(
                    x=x_coordinates,
                    y=y_coordinates,
                    z=progression_land_background,
                    colorscale=[
                        [0.0, "#CCCCCC"],
                        [1.0, "#CCCCCC"],
                    ],
                    zmin=0,
                    zmax=1,
                    showscale=False,
                    hoverinfo="skip",
                ),
                row=row_number,
                col=column_number,
            )

            block_start = (
                EVENT_DATE
                + pd.Timedelta(
                    days=(
                        block_number
                        * ROMAN_BLOCK_DAYS
                    )
                )
            )

            block_end = (
                block_start
                + pd.Timedelta(
                    days=(
                        ROMAN_BLOCK_DAYS
                        - 1
                    )
                )
            )

            if (
                composite_map is not None
                and np.isfinite(
                    composite_map.values
                ).any()
            ):
                figure.add_trace(
                    go.Heatmap(
                        x=composite_map["x"].values,
                        y=composite_map["y"].values,
                        z=composite_map.values,
                        coloraxis="coloraxis",
                        zsmooth=False,
                        hoverongaps=False,
                        hovertemplate=(
                            f"{row_label.replace('<br>', ' ')}"
                            "<br>"
                            f"Composite {column_number}"
                            "<br>"
                            f"{block_start:%d %b %Y}"
                            "–"
                            f"{block_end:%d %b %Y}"
                            "<br>"
                            "DNB-BRDF: "
                            "%{z:.2f} nW cm⁻² sr⁻¹"
                            "<extra></extra>"
                        ),
                    ),
                    row=row_number,
                    col=column_number,
                )

            else:
                figure.add_trace(
                    go.Scatter(
                        x=[longitude_centre],
                        y=[latitude_centre],
                        mode="text",
                        text=["No data"],
                        textfont=dict(
                            family="Arial",
                            size=9,
                            color="#666666",
                        ),
                        hoverinfo="skip",
                        showlegend=False,
                    ),
                    row=row_number,
                    col=column_number,
                )

            # Boundary overlay
            figure.add_trace(
                go.Scatter(
                    x=progression_boundary_x,
                    y=progression_boundary_y,
                    mode="lines",
                    line=dict(
                        color="#303030",
                        width=0.7,
                    ),
                    hoverinfo="skip",
                    showlegend=False,
                ),
                row=row_number,
                col=column_number,
            )

            figure.update_xaxes(
                range=list(
                    TACLOBAN_X_RANGE
                ),
                showticklabels=False,
                ticks="",
                showgrid=False,
                zeroline=False,
                fixedrange=True,
                row=row_number,
                col=column_number,
            )

            figure.update_yaxes(
                range=list(
                    TACLOBAN_Y_RANGE
                ),
                showticklabels=False,
                ticks="",
                showgrid=False,
                zeroline=False,
                fixedrange=True,
                row=row_number,
                col=column_number,
            )

    figure.add_annotation(
        x=0.5,
        y=-0.055,
        xref="paper",
        yref="paper",
        text=(
            "Sequential four-day composite "
            "within each phase"
        ),
        showarrow=False,
        font=dict(
            family="Arial",
            size=14,
            color="#243B5A",
        ),
    )

    figure.update_annotations(
        font=dict(
            family="Arial",
            size=11,
            color="#243B5A",
        ),
    )

    figure.update_layout(
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        width=2600,
        height=900,
        title=dict(
            text=figure_title,
            x=0.5,
            y=0.985,
            xanchor="center",
            font=dict(
                family="Arial",
                size=25,
                color="#243B5A",
            ),
        ),
        coloraxis=dict(
            colorscale="Inferno",
            cmin=0,
            cmax=progression_color_max,
            colorbar=dict(
                title=dict(
                    text=(
                        "DNB-BRDF"
                        "<br>"
                        "nW cm⁻² sr⁻¹"
                    )
                ),
                x=1.005,
                xanchor="left",
                y=0.5,
                len=0.86,
                thickness=20,
                outlinewidth=0.8,
                outlinecolor="#555555",
            ),
        ),
        font=dict(
            family="Arial",
            size=11,
            color="#243B5A",
        ),
        showlegend=False,
        margin=dict(
            l=120,
            r=145,
            t=105,
            b=70,
        ),
        hovermode="closest",
    )

    return figure


# ------------------------------------------------------------
# Román-style progression
# ------------------------------------------------------------

fig_roman_progression = (
    plot_composite_progression(
        map_grid=roman_progression_maps,
        figure_title=(
            "Román-style DNB-BRDF: "
            "four-day progression by phase"
        ),
    )
)

fig_roman_progression.show()


# ------------------------------------------------------------
# Reliability-qualified progression
# ------------------------------------------------------------

fig_rq_progression = (
    plot_composite_progression(
        map_grid=rq_progression_maps,
        figure_title=(
            "Reliability-qualified DNB-BRDF: "
            "four-day progression by phase"
        ),
    )
)

fig_rq_progression.show()


# ------------------------------------------------------------
# Optional static PNG export
# Both images use the same radiance scale.
# ------------------------------------------------------------

# fig_roman_progression.write_image(
#     "roman_15x4_progression.png",
#     scale=2,
# )

# fig_rq_progression.write_image(
#     "rq_15x4_progression.png",
#     scale=2,
# )

In [ ]:
# # ============================================================
# # 10.9 FULL SAMAR–LEYTE ROMÁN-STYLE PROGRESSION
# # ============================================================

# # Uses progression_rows defined in Section 10.8.


# # ------------------------------------------------------------
# # Construct the full regional land mask
# # ------------------------------------------------------------

# regional_reference = (
#     roman_composites
#     .isel(block=0)
# )

# regional_x_coordinates = (
#     regional_reference["x"].values
# )

# regional_y_coordinates = (
#     regional_reference["y"].values
# )

# regional_longitude_grid, regional_latitude_grid = (
#     np.meshgrid(
#         regional_x_coordinates,
#         regional_y_coordinates,
#     )
# )

# regional_pixel_centres = gpd.GeoSeries(
#     gpd.points_from_xy(
#         regional_longitude_grid.ravel(),
#         regional_latitude_grid.ravel(),
#     ),
#     crs="EPSG:4326",
# )

# regional_land_geometry = (
#     region_boundary
#     .geometry
#     .unary_union
# )

# regional_land_values = (
#     regional_pixel_centres
#     .intersects(regional_land_geometry)
#     .to_numpy()
#     .reshape(
#         len(regional_y_coordinates),
#         len(regional_x_coordinates),
#     )
# )

# regional_land_mask = xr.DataArray(
#     regional_land_values,
#     dims=("y", "x"),
#     coords={
#         "y": regional_y_coordinates,
#         "x": regional_x_coordinates,
#     },
# )

# # Román-style support:
# # baseline-valid land pixels, without GHSL filtering.
# roman_regional_mask = (
#     regional_land_mask
#     & roman_fixed_mask
#     .fillna(False)
#     .astype(bool)
# )

# regional_land_background = np.where(
#     regional_land_mask.values,
#     1.0,
#     np.nan,
# )


# # ------------------------------------------------------------
# # Prepare the 60 regional composite maps
# # ------------------------------------------------------------

# available_roman_blocks = set(
#     np.asarray(
#         roman_composites["block"].values,
#         dtype=int,
#     )
# )

# roman_regional_progression_maps = []

# for _, block_numbers in progression_rows:
#     map_row = []

#     for block_number in block_numbers:
#         if block_number not in available_roman_blocks:
#             map_row.append(None)
#             continue

#         composite_map = (
#             roman_composites
#             .sel(block=block_number)
#             .where(roman_regional_mask)
#         )

#         if hasattr(
#             composite_map.data,
#             "compute",
#         ):
#             composite_map = (
#                 composite_map.compute()
#             )

#         map_row.append(composite_map)

#     roman_regional_progression_maps.append(
#         map_row
#     )


# # ------------------------------------------------------------
# # Common radiance scale across all 60 maps
# # ------------------------------------------------------------

# regional_progression_values = []

# for map_row in roman_regional_progression_maps:
#     for composite_map in map_row:
#         if composite_map is None:
#             continue

#         values = np.asarray(
#             composite_map.values,
#             dtype=float,
#         )

#         values = values[
#             np.isfinite(values)
#         ]

#         if values.size > 0:
#             regional_progression_values.append(
#                 values
#             )

# if not regional_progression_values:
#     raise ValueError(
#         "No valid regional Román-style composites "
#         "were available."
#     )

# regional_progression_values = np.concatenate(
#     regional_progression_values
# )

# regional_progression_color_max = max(
#     float(
#         np.nanpercentile(
#             regional_progression_values,
#             98,
#         )
#     ),
#     1.0,
# )

# print(
#     "Regional progression radiance range: "
#     f"0–{regional_progression_color_max:.2f} "
#     "nW cm⁻² sr⁻¹"
# )


# # ------------------------------------------------------------
# # Regional boundary overlay
# # ------------------------------------------------------------

# regional_boundary_lines = (
#     extract_boundary_lines(
#         region_boundary
#     )
# )

# regional_boundary_x = []
# regional_boundary_y = []

# for longitude, latitude in regional_boundary_lines:
#     regional_boundary_x.extend(
#         [
#             *longitude.tolist(),
#             None,
#         ]
#     )

#     regional_boundary_y.extend(
#         [
#             *latitude.tolist(),
#             None,
#         ]
#     )


# # ------------------------------------------------------------
# # Plot the 4 × 15 regional grid
# # ------------------------------------------------------------

# fig_roman_regional_progression = make_subplots(
#     rows=4,
#     cols=15,
#     shared_xaxes=True,
#     shared_yaxes=True,
#     horizontal_spacing=0.002,
#     vertical_spacing=0.025,
#     column_titles=[
#         str(column_number)
#         for column_number in range(1, 16)
#     ],
# )

# regional_x_range = [
#     float(
#         np.nanmin(
#             regional_x_coordinates
#         )
#     ),
#     float(
#         np.nanmax(
#             regional_x_coordinates
#         )
#     ),
# ]

# regional_y_range = [
#     float(
#         np.nanmin(
#             regional_y_coordinates
#         )
#     ),
#     float(
#         np.nanmax(
#             regional_y_coordinates
#         )
#     ),
# ]

# regional_longitude_centre = np.mean(
#     regional_x_range
# )

# regional_latitude_centre = np.mean(
#     regional_y_range
# )

# for row_number, (
#     row_label,
#     block_numbers,
# ) in enumerate(
#     progression_rows,
#     start=1,
# ):
#     fig_roman_regional_progression.update_yaxes(
#         title_text=row_label,
#         title_font=dict(size=12),
#         title_standoff=4,
#         row=row_number,
#         col=1,
#     )

#     for column_number, block_number in enumerate(
#         block_numbers,
#         start=1,
#     ):
#         composite_map = (
#             roman_regional_progression_maps[
#                 row_number - 1
#             ][
#                 column_number - 1
#             ]
#         )

#         # Gray regional land background.
#         fig_roman_regional_progression.add_trace(
#             go.Heatmap(
#                 x=regional_x_coordinates,
#                 y=regional_y_coordinates,
#                 z=regional_land_background,
#                 colorscale=[
#                     [0.0, "#CCCCCC"],
#                     [1.0, "#CCCCCC"],
#                 ],
#                 zmin=0,
#                 zmax=1,
#                 showscale=False,
#                 hoverinfo="skip",
#             ),
#             row=row_number,
#             col=column_number,
#         )

#         block_start = (
#             EVENT_DATE
#             + pd.Timedelta(
#                 days=(
#                     block_number
#                     * ROMAN_BLOCK_DAYS
#                 )
#             )
#         )

#         block_end = (
#             block_start
#             + pd.Timedelta(
#                 days=(
#                     ROMAN_BLOCK_DAYS - 1
#                 )
#             )
#         )

#         if (
#             composite_map is not None
#             and np.isfinite(
#                 composite_map.values
#             ).any()
#         ):
#             fig_roman_regional_progression.add_trace(
#                 go.Heatmap(
#                     x=composite_map["x"].values,
#                     y=composite_map["y"].values,
#                     z=composite_map.values,
#                     coloraxis="coloraxis",
#                     zsmooth=False,
#                     hoverongaps=False,
#                     hovertemplate=(
#                         f"Composite {column_number}<br>"
#                         f"{block_start:%d %b %Y}"
#                         "–"
#                         f"{block_end:%d %b %Y}<br>"
#                         "DNB-BRDF: "
#                         "%{z:.2f} nW cm⁻² sr⁻¹"
#                         "<extra></extra>"
#                     ),
#                 ),
#                 row=row_number,
#                 col=column_number,
#             )

#         else:
#             fig_roman_regional_progression.add_trace(
#                 go.Scatter(
#                     x=[
#                         regional_longitude_centre
#                     ],
#                     y=[
#                         regional_latitude_centre
#                     ],
#                     mode="text",
#                     text=["No data"],
#                     textfont=dict(
#                         size=8,
#                         color="#666666",
#                     ),
#                     hoverinfo="skip",
#                     showlegend=False,
#                 ),
#                 row=row_number,
#                 col=column_number,
#             )

#         fig_roman_regional_progression.add_trace(
#             go.Scatter(
#                 x=regional_boundary_x,
#                 y=regional_boundary_y,
#                 mode="lines",
#                 line=dict(
#                     color="#303030",
#                     width=0.6,
#                 ),
#                 hoverinfo="skip",
#                 showlegend=False,
#             ),
#             row=row_number,
#             col=column_number,
#         )

#         fig_roman_regional_progression.update_xaxes(
#             range=regional_x_range,
#             showticklabels=False,
#             ticks="",
#             showgrid=False,
#             zeroline=False,
#             fixedrange=True,
#             row=row_number,
#             col=column_number,
#         )

#         fig_roman_regional_progression.update_yaxes(
#             range=regional_y_range,
#             showticklabels=False,
#             ticks="",
#             showgrid=False,
#             zeroline=False,
#             fixedrange=True,
#             row=row_number,
#             col=column_number,
#         )

# fig_roman_regional_progression.add_annotation(
#     x=0.5,
#     y=-0.055,
#     xref="paper",
#     yref="paper",
#     text=(
#         "Sequential four-day composite "
#         "within each phase"
#     ),
#     showarrow=False,
#     font=dict(
#         family="Arial",
#         size=14,
#         color="#243B5A",
#     ),
# )

# fig_roman_regional_progression.update_annotations(
#     font=dict(
#         family="Arial",
#         size=11,
#         color="#243B5A",
#     ),
# )

# fig_roman_regional_progression.update_layout(
#     template="plotly_white",
#     paper_bgcolor="rgba(0,0,0,0)",
#     plot_bgcolor="white",
#     width=2800,
#     height=1100,
#     title=dict(
#         text=(
#             "Román-style DNB-BRDF: "
#             "Samar–Leyte four-day progression"
#         ),
#         x=0.5,
#         y=0.985,
#         xanchor="center",
#         font=dict(size=25),
#     ),
#     coloraxis=dict(
#         colorscale="Inferno",
#         cmin=0,
#         cmax=regional_progression_color_max,
#         colorbar=dict(
#             title=dict(
#                 text=(
#                     "DNB-BRDF"
#                     "<br>"
#                     "nW cm⁻² sr⁻¹"
#                 )
#             ),
#             x=1.005,
#             xanchor="left",
#             y=0.5,
#             len=0.86,
#             thickness=20,
#             outlinewidth=0.8,
#             outlinecolor="#555555",
#         ),
#     ),
#     font=dict(
#         family="Arial",
#         size=11,
#         color="#243B5A",
#     ),
#     showlegend=False,
#     margin=dict(
#         l=125,
#         r=145,
#         t=110,
#         b=75,
#     ),
# )

# fig_roman_regional_progression.show()

# # Optional PNG export
# # fig_roman_regional_progression.write_image(
# #     "roman_samar_leyte_15x4_progression.png",
# #     scale=2,
# # )

In [ ]:
# ============================================================
# 10.10 FOUR-DAY COMPOSITE CONSTRUCTION EXAMPLE
# ============================================================

EXAMPLE_BLOCK = 0

example_start_date = (
    EVENT_DATE
    + pd.Timedelta(
        days=(
            EXAMPLE_BLOCK
            * ROMAN_BLOCK_DAYS
        )
    )
)

example_dates = pd.DatetimeIndex(
    [
        example_start_date
        + pd.Timedelta(days=day_offset)
        for day_offset in range(
            ROMAN_BLOCK_DAYS
        )
    ]
)

print(
    "Example period:",
    example_dates[0].date(),
    "to",
    example_dates[-1].date(),
)


# ------------------------------------------------------------
# Extract the four direct daily observations
# ------------------------------------------------------------

example_daily_cube = (
    roman_cube
    .reindex(date=example_dates)
    .where(roman_regional_mask)
)

if hasattr(
    example_daily_cube.data,
    "compute",
):
    example_daily_cube = (
        example_daily_cube.compute()
    )


# ------------------------------------------------------------
# Reconstruct the composite directly from the four days
# ------------------------------------------------------------

example_valid_day_count = (
    example_daily_cube
    .notnull()
    .sum(dim="date")
)

example_composite_calculated = (
    example_daily_cube
    .mean(
        dim="date",
        skipna=True,
    )
)


# Stored composite produced by the main workflow
example_composite_stored = (
    roman_composites
    .sel(block=EXAMPLE_BLOCK)
    .where(roman_regional_mask)
)

if hasattr(
    example_composite_stored.data,
    "compute",
):
    example_composite_stored = (
        example_composite_stored.compute()
    )


# ------------------------------------------------------------
# Confirm that both calculations match
# ------------------------------------------------------------

comparison_values = np.asarray(
    (
        example_composite_calculated
        - example_composite_stored
    ).values,
    dtype=float,
)

finite_comparison = comparison_values[
    np.isfinite(comparison_values)
]

maximum_difference = (
    float(
        np.nanmax(
            np.abs(
                finite_comparison
            )
        )
    )
    if finite_comparison.size > 0
    else np.nan
)

print(
    "Maximum difference from stored composite:",
    f"{maximum_difference:.10f}",
)

for valid_day_number in range(5):
    pixel_count = int(
        (
            example_valid_day_count
            == valid_day_number
        )
        .sum()
        .item()
    )

    print(
        f"Pixels with {valid_day_number} "
        f"valid day(s): {pixel_count:,}"
    )


# ------------------------------------------------------------
# Shared example radiance scale
# ------------------------------------------------------------

example_values = []

for date_value in example_dates:
    daily_values = np.asarray(
        example_daily_cube
        .sel(date=date_value)
        .values,
        dtype=float,
    )

    daily_values = daily_values[
        np.isfinite(daily_values)
    ]

    if daily_values.size > 0:
        example_values.append(
            daily_values
        )

composite_values = np.asarray(
    example_composite_calculated.values,
    dtype=float,
)

composite_values = composite_values[
    np.isfinite(composite_values)
]

if composite_values.size > 0:
    example_values.append(
        composite_values
    )

if not example_values:
    raise ValueError(
        "No valid observations were available "
        "for the selected example block."
    )

example_values = np.concatenate(
    example_values
)

example_color_max = max(
    float(
        np.nanpercentile(
            example_values,
            98,
        )
    ),
    1.0,
)


# ------------------------------------------------------------
# Plot four daily maps and the resulting mean composite
# ------------------------------------------------------------

example_titles = [
    date_value.strftime(
        "%d %b %Y"
    )
    for date_value in example_dates
] + [
    "Four-day mean",
]

fig_composite_example = make_subplots(
    rows=1,
    cols=5,
    shared_xaxes=True,
    shared_yaxes=True,
    horizontal_spacing=0.012,
    column_titles=example_titles,
)

example_maps = [
    example_daily_cube.sel(
        date=date_value
    )
    for date_value in example_dates
] + [
    example_composite_calculated,
]

for column_number, example_map in enumerate(
    example_maps,
    start=1,
):
    # Gray land background
    fig_composite_example.add_trace(
        go.Heatmap(
            x=regional_x_coordinates,
            y=regional_y_coordinates,
            z=regional_land_background,
            colorscale=[
                [0.0, "#CCCCCC"],
                [1.0, "#CCCCCC"],
            ],
            zmin=0,
            zmax=1,
            showscale=False,
            hoverinfo="skip",
        ),
        row=1,
        col=column_number,
    )

    fig_composite_example.add_trace(
        go.Heatmap(
            x=example_map["x"].values,
            y=example_map["y"].values,
            z=example_map.values,
            coloraxis="coloraxis",
            zsmooth=False,
            hoverongaps=False,
            hovertemplate=(
                "DNB-BRDF: "
                "%{z:.2f} nW cm⁻² sr⁻¹"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=column_number,
    )

    fig_composite_example.add_trace(
        go.Scatter(
            x=regional_boundary_x,
            y=regional_boundary_y,
            mode="lines",
            line=dict(
                color="#303030",
                width=0.8,
            ),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=1,
        col=column_number,
    )

    fig_composite_example.update_xaxes(
        range=regional_x_range,
        showticklabels=False,
        ticks="",
        showgrid=False,
        zeroline=False,
        fixedrange=True,
        row=1,
        col=column_number,
    )

    fig_composite_example.update_yaxes(
        range=regional_y_range,
        showticklabels=False,
        ticks="",
        showgrid=False,
        zeroline=False,
        fixedrange=True,
        row=1,
        col=column_number,
    )

fig_composite_example.add_annotation(
    x=0.805,
    y=0.5,
    xref="paper",
    yref="paper",
    text="→",
    showarrow=False,
    font=dict(
        size=30,
        color="#0057FF",
    ),
)

fig_composite_example.update_annotations(
    font=dict(
        family="Arial",
        size=14,
        color="#243B5A",
    ),
)

fig_composite_example.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1700,
    height=540,
    title=dict(
        text=(
            "Four daily DNB-BRDF observations "
            "combined into one four-day composite"
        ),
        x=0.5,
        y=0.98,
        xanchor="center",
        font=dict(size=24),
    ),
    coloraxis=dict(
        colorscale="Inferno",
        cmin=0,
        cmax=example_color_max,
        colorbar=dict(
            title=dict(
                text=(
                    "DNB-BRDF"
                    "<br>"
                    "nW cm⁻² sr⁻¹"
                )
            ),
            x=1.01,
            xanchor="left",
            y=0.5,
            len=0.86,
            thickness=20,
            outlinewidth=0.8,
            outlinecolor="#555555",
        ),
    ),
    font=dict(
        family="Arial",
        size=13,
        color="#243B5A",
    ),
    showlegend=False,
    margin=dict(
        l=40,
        r=135,
        t=100,
        b=35,
    ),
)

fig_composite_example.show()

# Optional PNG export
# fig_composite_example.write_image(
#     "four_day_composite_example.png",
#     scale=2,
# )

### Method: Román-style transfer and reliability-qualified comparison

This experiment transfers the relative-recovery approach of [Román et al. (2019)](https://doi.org/10.1371/journal.pone.0218883) to Samar–Leyte and compares it with a reliability-qualified implementation. Both workflows use the same temporal aggregation, baseline period, recovery calculation, spatial-completeness threshold, and NGCP comparator so that their resulting trajectories are directly comparable.

Daily observations were grouped into non-overlapping four-day mean composites. A 60-day pre-Haiyan period was used to estimate the baseline because the exact baseline window in Román et al. was not sufficiently specified for direct reproduction. For each pixel, baseline radiance \(NTL_{p,0}\) was calculated as the median of its available pre-Haiyan four-day composites. Regional recovery for composite \(i\) was then calculated using identical valid-pixel support in the numerator and denominator:

$ Recovery_i = 100 \frac{\sum_{p \in V_i} NTL_{p,i}} {\sum_{p \in V_i} NTL_{p,0}},$

where $V_i$ contains pixels with both a valid composite observation and a valid baseline. Four-day composites representing less than 10% of the fixed baseline-lit spatial support were treated as inadmissible.

The two NTL implementations differ only in their input and reliability treatment:

- **Román-style transfer:** gap-filled DNB-BRDF radiance over all baseline-lit land pixels, with basic fill-value screening but no GHSL or mandatory-quality filtering. The gap-filled layer is used as the closest available analogue to the cloud-free Black Marble observations described by Román et al.
- **Reliability-qualified NTL:** directly observed DNB-BRDF radiance restricted to `MQF == 0` and GHSL G7 settlement pixels (classes 23 and 30), with daily values clamped at the spatial 95th percentile. No moving average was applied.

NGCP Leyte–Samar Hour 1 demand was aggregated into the same four-day periods and expressed relative to its median over the same 60-day baseline. NGCP demand is treated as an independent functional comparator rather than a direct measure of the percentage of customers reconnected.

For spatial interpretation, the baseline and median radiance during three fixed post-Haiyan intervals were mapped over Tacloban: 0–59 days, 60–119 days, and 120–179 days. These intervals reproduce the approximate duration structure used by Román et al.; they are not assumed to represent equivalent institutional recovery phases in the Philippine context. All maps use one radiance scale. The Román row displays all land pixels, whereas the reliability-qualified row displays only GHSL G7 pixels; gray cells denote non-G7 land and white areas denote water.


### Preliminary results and discussion

Both NTL implementations detect a marked reduction following Haiyan and a subsequent long-term increase, indicating that the broad disruption–recovery pattern is transferable to Samar–Leyte. However, the magnitude, continuity, and stability of the estimated trajectory depend strongly on preprocessing and observation support.

The Román-style profile detects the post-Haiyan decline but remains substantially brighter than the NGCP trajectory during the acute outage period. NGCP demand falls to approximately zero, whereas gap-filled NTL initially remains around 50–80% of its baseline. This divergence suggests that gap filling preserves or reconstructs residual radiance during periods when directly observed lighting is limited. Remaining light may also represent generators, priority facilities, temporary response activity, or outdoor lighting that is not proportional to regional electricity demand.

The Román-style series also exhibits considerable pre-event variability and several post-event values above 100%. Recovery above 100% is mathematically possible because it indicates radiance exceeding the pixel-specific baseline. Nevertheless, repeated increases above approximately 150–250% are unlikely to represent electricity restoration alone. They may reflect gap-filling behaviour, changing spatial support, seasonal or atmospheric effects, temporary lighting, or changes in the composition of observable light sources. Consequently, the gap-filled profile captures the general direction of recovery but does not provide a consistently interpretable estimate of its magnitude.

The reliability-qualified profile reproduces the immediate collapse more closely, falling to approximately 15–20% of baseline immediately after Haiyan. Its subsequent increase also broadly follows the direction of NGCP recovery. During the first 60 days, both series remain strongly suppressed; during the second and third intervals, both show progressive recovery, although the NTL trajectory is more variable and frequently exceeds the relative NGCP load.

This improvement comes with reduced temporal completeness. Several four-day periods are absent because insufficient directly observed `MQF == 0` pixels remain after GHSL and spatial-completeness filtering. The remaining pre-Haiyan RQ composites also vary substantially despite satisfying the 10% threshold. Therefore, 10% should be interpreted as a minimum admissibility condition, not evidence that every retained composite is precise. Spatial coverage and the number of contributing observations should remain visible alongside any reported recovery estimate.

The Tacloban maps support the temporal interpretation. Both methods identify a bright pre-Haiyan urban and coastal corridor followed by widespread dimming during the first 60 days. The reliability-qualified maps show a sharper reduction within the G7 settlement support, while the gap-filled maps retain more residual brightness. Stage 2 shows spatially uneven partial recovery, with lighting returning first in selected parts of the urban corridor rather than uniformly across Tacloban. By Stage 3, radiance has expanded and intensified but remains spatially heterogeneous relative to the baseline.

The shared color scale confirms that the apparent changes represent differences in radiance rather than panel-specific rescaling. However, the spatial extent of the two rows should not be compared as equivalent pixel counts: the Román-style maps include all baseline-lit land pixels, whereas the RQ maps intentionally retain only GHSL G7 settlement pixels. Gray RQ cells are excluded land and must not be interpreted as zero radiance or continued outage.

Overall, the Román framework is transferable for detecting the event and describing the broad direction of recovery, but its gap-filled whole-region trajectory is unstable as a quantitative estimate of recovery magnitude under Samar–Leyte observation conditions. Reliability qualification produces a more plausible acute-impact signal and closer directional agreement with electricity demand, but at the cost of substantial gaps and remaining four-day variability. This supports treating observability as a prerequisite for recovery inference rather than assuming that temporal compositing alone resolves tropical cloud limitations.

The comparison remains a workflow-level test. Because the two implementations differ simultaneously in radiance source, MQF filtering, GHSL support, and percentile clipping, any improvement cannot yet be attributed to a single processing step. In addition, NGCP load and satellite radiance measure related but non-identical processes: load represents aggregate electricity demand, while NTL primarily reflects upward-emitted nighttime lighting. Their divergence is therefore analytically meaningful and should not automatically be treated as satellite error.
